# Stop-loss backtest

This notebook compares selling a stock after a preset loss and reinvesting in SPY against holding the stock to the end of the same period.

## Backtest setup

- **Universe:** current U.S.-listed equities from Nasdaq Trader's listing files, excluding ETFs and other non-standard instruments where identifiable. Histories are fetched through `yfinance`.
- **Entry:** first available closing price of each stock in each calendar month.
- **Loss thresholds:** 5%, 10%, 15%, 20%, 30%, 40%, and 50% below the entry price. These are fixed entry-price stops, not trailing stops.
- **Trigger and trade:** a daily closing price crosses the threshold; the stock is sold at its next available opening price and the proceeds are allocated to SPY at that open.
- **Comparison:** the stop strategy's terminal return minus the same stock's buy-and-hold return.
- **Holding periods:** 21, 63, 126, 252, and 756 SPY sessions (approximately 1 month, 3 months, 6 months, 1 year, and 3 years).
- **Primary comparison:** a 20% stop over 252 sessions.

Returns use adjusted prices. Stops use Yahoo's split-adjusted, non-dividend-adjusted `Close`. With `auto_adjust=False`, Yahoo already adjusts `Open` and `Close` for splits; applying another split adjustment would introduce false triggers. Historical as-traded entry prices are reconstructed separately using subsequent split ratios, solely for the $1 and $5 entry-price screens. The notebook checks this convention with both a synthetic example and NVDA's June 2024 split.

## Sensitivity analyses

The notebook reports event-, ticker-, calendar-year-, and ticker-year-weighted estimates; ticker and year-block bootstrap intervals; entry-price screens; time-period comparisons; and tail and rebound diagnostics. Price-history quality is assessed using documented flags, with raw, hard-exclusion, and broader review-exclusion samples shown separately.

The symbol list is drawn from **currently listed** stocks, so this is not a survivorship-free historical universe. Delisted stocks are missing; the tipping-point exercise only measures sensitivity to that omission. Monthly observation windows overlap, and transaction costs, slippage, and taxes are not modeled.

Run from top to bottom. Outputs have been cleared from the public notebook. Local Yahoo price caches and event-level files are not distributed.


In [ ]:
#%pip install -q yfinance pandas numpy matplotlib pyarrow duckdb

In [ ]:
from pathlib import Path
import time
import warnings

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RNG = np.random.default_rng(42)

WORKDIR = Path("sell_timing_work")
PRICE_CACHE = WORKDIR / "yf_price_cache"
# v5: new directory so event parts built with the old (double split-adjusted) trigger are never reused.
# The old WORKDIR / "events" folder can be deleted once you are happy with v5.
EVENT_DIR = WORKDIR / "events_v5"
AGG_DIR = Path("sell_timing_aggregates")

for p in [WORKDIR, PRICE_CACHE, EVENT_DIR, AGG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Keep locally generated Yahoo-derived data out of Git even if the parent directory is added.
(WORKDIR / ".gitignore").write_text("*\n!.gitignore\n", encoding="utf-8")

print("yfinance:", yf.__version__)
print("pandas:", pd.__version__)
print("work directory:", WORKDIR.resolve())

## 1. Configuration

The defaults run the full analysis. For a quick dry run, set `MAX_TICKERS` to something like `100`.

`YF_REPAIR=True` asks yfinance to repair known Yahoo price errors. The data-access logic is isolated so a future commercially licensed provider can replace Yahoo without rewriting the analysis.

In [ ]:
THRESHOLDS = np.array([-0.05, -0.10, -0.15, -0.20, -0.30, -0.40, -0.50], dtype=float)

HORIZONS = {
    "1m": 21,
    "3m": 63,
    "6m": 126,
    "1y": 252,
    "3y": 756,
}

PRIMARY_THRESHOLD = -0.20
PRIMARY_HORIZON_DAYS = 252

# None = all current-listed symbols that survive the universe filters.
MAX_TICKERS = None

BATCH_SIZE = 50
YF_REPAIR = True
OVERWRITE_PRICE_CACHE = False
REBUILD_EVENTS = False

# Metadata is optional because per-ticker Yahoo metadata calls are much slower and more fragile
# than the historical-price download. Turn on if you want sector/industry figures.
FETCH_SECTOR_INDUSTRY = False
METADATA_SLEEP_SECONDS = 0.05

N_BOOT = 2_000
PRICE_FLOORS = [0.0, 1.0, 5.0]

# Used only for the survivorship tipping-point diagnostic.
CATASTROPHIC_STOP_ADVANTAGES = [0.25, 0.50, 0.75, 1.00]

print("thresholds:", THRESHOLDS)
print("horizons:", HORIZONS)

## 2. Current U.S.-listed equity universe

Nasdaq Trader's symbol directories are used only to define the current-listed universe. ETFs, test issues, warrants, rights, units, preferreds, debt-like securities, and obviously non-standard symbols are excluded.

This is deliberately a **current-listed universe**, not a historical constituent database. That maximizes breadth with freely accessible data but creates survivorship bias, handled later as a limitation and tipping-point analysis.

In [ ]:
NASDAQ_LISTED_URL = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt"
OTHER_LISTED_URL = "https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt"

EXCLUDE_NAME_RE = (
    r"\b("
    r"Warrant|Warrants|Right|Rights|Unit|Units|"
    r"Preferred|Preference|"
    r"Note|Notes|Debenture|Debentures|Bond|Bonds|"
    r"Exchange Traded Note|ETN"
    r")\b"
)

def _normalize_yahoo_symbol(s: str) -> str:
    s = str(s).strip().upper()
    # Yahoo uses '-' for share-class dots such as BRK.B -> BRK-B.
    return s.replace(".", "-")

def load_current_us_equity_universe() -> pd.DataFrame:
    nas = pd.read_csv(NASDAQ_LISTED_URL, sep="|")
    oth = pd.read_csv(OTHER_LISTED_URL, sep="|")

    nas = nas.rename(columns={"Symbol": "symbol"})
    oth = oth.rename(columns={"ACT Symbol": "symbol"})

    nas["exchange"] = "NASDAQ"
    oth["exchange"] = oth["Exchange"].map({
        "N": "NYSE",
        "A": "NYSE American",
        "P": "NYSE Arca",
        "Z": "Cboe BZX",
        "V": "IEX",
    }).fillna(oth["Exchange"].astype(str))

    keep_cols = ["symbol", "Security Name", "ETF", "Test Issue", "exchange"]
    nas = nas[[c for c in keep_cols if c in nas.columns]].copy()
    oth = oth[[c for c in keep_cols if c in oth.columns]].copy()

    u = pd.concat([nas, oth], ignore_index=True)
    u = u.rename(columns={"Security Name": "security_name"})

    u["symbol"] = u["symbol"].astype(str).str.strip()
    u["security_name"] = u["security_name"].astype(str).str.strip()

    # Remove footer rows and non-real/test listings.
    u = u[u["symbol"].ne("") & ~u["symbol"].str.contains("File Creation Time", case=False, na=False)]
    if "ETF" in u.columns:
        u = u[u["ETF"].fillna("N").eq("N")]
    if "Test Issue" in u.columns:
        u = u[u["Test Issue"].fillna("N").eq("N")]

    u = u[~u["security_name"].str.contains(EXCLUDE_NAME_RE, case=False, regex=True, na=False)]

    # Keep symbols Yahoo can reasonably resolve without special preferred/warrant syntax.
    u = u[u["symbol"].str.match(r"^[A-Z0-9.\-]+$", na=False)]
    u["ticker"] = u["symbol"].map(_normalize_yahoo_symbol)

    u = (
        u[["ticker", "symbol", "security_name", "exchange"]]
        .drop_duplicates("ticker")
        .sort_values("ticker")
        .reset_index(drop=True)
    )

    if MAX_TICKERS is not None:
        u = u.head(int(MAX_TICKERS)).copy()

    return u

universe = load_current_us_equity_universe()
universe.to_csv(WORKDIR / "universe_snapshot.csv", index=False)

print(f"Universe size: {len(universe):,}")
display(universe.head(10))
display(universe["exchange"].value_counts(dropna=False).to_frame("n"))

## 3. Download and cache prices

`get_price_history()` and `download_missing_price_history()` handle price retrieval and caching. The rest of the analysis expects the columns and split-adjustment conventions documented below. Downloaded data stays local.


In [ ]:
PRICE_COLS = ["Open", "High", "Low", "Close", "Adj Close", "Volume", "Dividends", "Stock Splits"]

def _cache_path(ticker: str) -> Path:
    return PRICE_CACHE / f"{ticker}.parquet"

def _empty_marker(ticker: str) -> Path:
    return PRICE_CACHE / f"{ticker}.empty"

def _clean_download_frame(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=PRICE_COLS)

    out = df.copy()
    out.index = pd.to_datetime(out.index, errors="coerce")
    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_localize(None)
    out = out[~out.index.isna()].sort_index()
    out = out[~out.index.duplicated(keep="last")]

    for c in PRICE_COLS:
        if c not in out.columns:
            out[c] = 0.0 if c in ["Dividends", "Stock Splits"] else np.nan

    out = out[PRICE_COLS]
    for c in PRICE_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    return out

def _extract_from_batch(raw: pd.DataFrame, ticker: str) -> pd.DataFrame:
    if raw is None or raw.empty:
        return pd.DataFrame(columns=PRICE_COLS)

    if not isinstance(raw.columns, pd.MultiIndex):
        return _clean_download_frame(raw)

    lv0 = raw.columns.get_level_values(0).astype(str)
    lv1 = raw.columns.get_level_values(1).astype(str)

    if ticker in set(lv0):
        sub = raw.xs(ticker, axis=1, level=0, drop_level=True)
    elif ticker in set(lv1):
        sub = raw.xs(ticker, axis=1, level=1, drop_level=True)
    else:
        return pd.DataFrame(columns=PRICE_COLS)

    return _clean_download_frame(sub)

def _download_batch(tickers):
    return yf.download(
        tickers=list(tickers),
        period="max",
        interval="1d",
        actions=True,
        auto_adjust=False,
        repair=YF_REPAIR,
        group_by="ticker",
        threads=True,
        progress=False,
        multi_level_index=True,
    )

def download_missing_price_history(tickers):
    tickers = list(dict.fromkeys(map(str, tickers)))
    missing = [
        t for t in tickers
        if OVERWRITE_PRICE_CACHE or (not _cache_path(t).exists() and not _empty_marker(t).exists())
    ]

    print(f"Need to download: {len(missing):,} / {len(tickers):,} tickers")

    for start in range(0, len(missing), BATCH_SIZE):
        batch = missing[start:start + BATCH_SIZE]
        print(f"Batch {start // BATCH_SIZE + 1:,}: {batch[0]} ... {batch[-1]} ({len(batch)} tickers)")

        try:
            raw = _download_batch(batch)
        except Exception as e:
            print("  batch download failed:", repr(e))
            raw = pd.DataFrame()

        failed = []
        for ticker in batch:
            frame = _extract_from_batch(raw, ticker)
            if len(frame) >= 2 and frame["Close"].notna().sum() >= 2:
                frame.to_parquet(_cache_path(ticker))
                _empty_marker(ticker).unlink(missing_ok=True)
            else:
                failed.append(ticker)

        # One-ticker fallback for symbols omitted from a batch response.
        for ticker in failed:
            try:
                one = _download_batch([ticker])
                frame = _extract_from_batch(one, ticker)
            except Exception:
                frame = pd.DataFrame(columns=PRICE_COLS)

            if len(frame) >= 2 and frame["Close"].notna().sum() >= 2:
                frame.to_parquet(_cache_path(ticker))
                _empty_marker(ticker).unlink(missing_ok=True)
            else:
                _empty_marker(ticker).write_text("", encoding="utf-8")

def get_price_history(ticker: str) -> pd.DataFrame:
    """Provider-isolated historical-price accessor used by the analysis."""
    p = _cache_path(ticker)
    if not p.exists() and not _empty_marker(ticker).exists():
        download_missing_price_history([ticker])

    if not p.exists():
        return pd.DataFrame(columns=PRICE_COLS)

    return _clean_download_frame(pd.read_parquet(p))

# SPY is the post-stop opportunity-cost benchmark.
download_missing_price_history(["SPY"])
spy_raw = get_price_history("SPY")
print("SPY rows:", len(spy_raw), "|", spy_raw.index.min(), "to", spy_raw.index.max())

In [ ]:
# Run the bulk download. Rerunning the notebook uses the cache.
download_missing_price_history(universe["ticker"])

downloaded = universe["ticker"].map(lambda t: _cache_path(t).exists())
print(f"Usable cached histories: {downloaded.sum():,} / {len(universe):,}")
print(f"Unavailable/empty: {(~downloaded).sum():,}")

## 4. Price preparation

Two different price concepts are needed:

1. **Stop trigger:** split-adjusted price, but not dividend-adjusted. This avoids falsely triggering on stock splits while preserving the idea of a literal price-loss threshold. With `auto_adjust=False`, Yahoo's `Close` is **already** exactly this series, so it is used directly.
2. **Portfolio return:** Yahoo adjusted prices. This gives a total-return-like comparison that incorporates split/dividend adjustments.

The entry-price floor uses the **as-traded close**, reconstructed as Yahoo `Close` × the product of all split ratios that occur *after* that date (Yahoo divides earlier prices by those ratios, so multiplying undoes it).

In [ ]:
def prepare_spy(spy: pd.DataFrame) -> pd.DataFrame:
    s = spy.copy().sort_index()
    s = s.loc[s["Close"].notna() & s["Adj Close"].notna()].copy()

    factor = s["Adj Close"] / s["Close"]
    s["adj_open"] = s["Open"] * factor
    s["adj_close"] = s["Adj Close"]

    s = s[["adj_open", "adj_close"]].replace([np.inf, -np.inf], np.nan)
    s = s.dropna()
    return s

SPY = prepare_spy(spy_raw)
SPY_DATES = SPY.index

def _future_split_factor(split_series: pd.Series) -> pd.Series:
    """
    Product of all split ratios recorded strictly after each date.
    Yahoo's split-adjusted price = as-traded price / this factor, so
    as-traded price = Yahoo Close x this factor.
    A split recorded on date t affects prices strictly before t, not t itself.
    """
    ratio = pd.to_numeric(split_series, errors="coerce").fillna(0.0)
    ratio = ratio.where(ratio > 0, 1.0)
    reverse_product = ratio.iloc[::-1].cumprod().iloc[::-1]
    return reverse_product.shift(-1, fill_value=1.0)

def prepare_stock_on_spy_calendar(raw: pd.DataFrame) -> pd.DataFrame:
    if raw.empty:
        return pd.DataFrame(index=SPY_DATES)

    r = raw.sort_index()

    # Compute the split factor on the stock's own full history, then align to the SPY calendar,
    # so no split row can be lost in the reindex.
    split_factor = _future_split_factor(r["Stock Splits"]).reindex(SPY_DATES)

    x = r.reindex(SPY_DATES)

    # yfinance with auto_adjust=False: Open/Close are ALREADY split-adjusted (not dividend-adjusted).
    yahoo_close = pd.to_numeric(x["Close"], errors="coerce")
    yahoo_open = pd.to_numeric(x["Open"], errors="coerce")
    adj_close = pd.to_numeric(x["Adj Close"], errors="coerce")

    adj_factor = adj_close / yahoo_close

    out = pd.DataFrame(index=SPY_DATES)
    out["raw_open"] = yahoo_open * split_factor     # as-traded open
    out["raw_close"] = yahoo_close * split_factor   # as-traded close (used for the $1/$5 floor)
    out["adj_open"] = yahoo_open * adj_factor
    out["adj_close"] = adj_close
    out["split_close"] = yahoo_close                # stop-trigger series

    return out.replace([np.inf, -np.inf], np.nan)

## 5. Event engine

For every stock-month entry:

- buy at that month's first observed close;
- look forward for the first close that breaches each stop threshold;
- if breached, execute at the stock's next available open;
- move proceeds immediately into SPY at the same market open;
- compare terminal wealth with simply holding the stock.

Horizons are counted in **SPY trading days**, so a "1 year" horizon is 252 market sessions rather than 252 observations for the individual stock.

In [ ]:
EVENT_COLUMNS = [
    "ticker", "entry_date", "entry_year", "raw_entry_close",
    "threshold", "horizon_label", "horizon_days",
    "hold_return", "strategy_return", "delta",
    "triggered", "trigger_date", "execution_date",
    "days_to_trigger", "days_to_execution",
]

def _strictly_next_valid_positions(values: np.ndarray) -> np.ndarray:
    """For each i, index of next finite positive value strictly after i, else -1."""
    n = len(values)
    out = np.full(n, -1, dtype=np.int32)
    next_seen = -1
    for i in range(n - 1, -1, -1):
        out[i] = next_seen
        if np.isfinite(values[i]) and values[i] > 0:
            next_seen = i
    return out

def simulate_ticker(ticker: str, raw: pd.DataFrame) -> pd.DataFrame:
    x = prepare_stock_on_spy_calendar(raw)
    if x.empty:
        return pd.DataFrame(columns=EVENT_COLUMNS)

    raw_close = x["raw_close"].to_numpy(float)
    adj_open = x["adj_open"].to_numpy(float)
    adj_close = x["adj_close"].to_numpy(float)
    split_close = x["split_close"].to_numpy(float)

    spy_open = SPY["adj_open"].to_numpy(float)
    spy_close = SPY["adj_close"].to_numpy(float)
    dates = SPY.index

    valid_entry = (
        np.isfinite(raw_close) & (raw_close > 0) &
        np.isfinite(adj_close) & (adj_close > 0) &
        np.isfinite(split_close) & (split_close > 0)
    )
    valid_pos = np.flatnonzero(valid_entry)
    if len(valid_pos) == 0:
        return pd.DataFrame(columns=EVENT_COLUMNS)

    valid_months = dates[valid_pos].to_period("M")
    first_of_month = np.r_[True, valid_months[1:] != valid_months[:-1]]
    entries = valid_pos[first_of_month]

    next_open_pos = _strictly_next_valid_positions(adj_open)
    max_h = max(HORIZONS.values())
    rows = []

    for e in entries:
        entry_adj = adj_close[e]
        entry_split = split_close[e]
        entry_raw = raw_close[e]

        if not (np.isfinite(entry_adj) and entry_adj > 0 and np.isfinite(entry_split) and entry_split > 0):
            continue

        max_end = min(e + max_h, len(dates) - 1)
        if max_end <= e:
            continue

        future = split_close[e + 1:max_end + 1] / entry_split - 1.0

        first_trigger = {}
        for threshold in THRESHOLDS:
            hits = np.flatnonzero(np.isfinite(future) & (future <= threshold))
            first_trigger[float(threshold)] = (e + 1 + int(hits[0])) if len(hits) else -1

        for label, h in HORIZONS.items():
            end = e + h
            if end >= len(dates):
                continue

            terminal_stock = adj_close[end]
            if not (np.isfinite(terminal_stock) and terminal_stock > 0):
                continue

            hold_return = terminal_stock / entry_adj - 1.0

            for threshold in THRESHOLDS:
                t = first_trigger[float(threshold)]
                exec_pos = next_open_pos[t] if t >= 0 else -1

                # A stop only changes this horizon if the next-open execution occurs
                # on or before the horizon date.
                triggered = int(t >= 0 and exec_pos >= 0 and exec_pos <= end)

                if triggered:
                    stock_exec_open = adj_open[exec_pos]
                    spy_exec_open = spy_open[exec_pos]
                    spy_terminal = spy_close[end]

                    if (
                        np.isfinite(stock_exec_open) and stock_exec_open > 0 and
                        np.isfinite(spy_exec_open) and spy_exec_open > 0 and
                        np.isfinite(spy_terminal) and spy_terminal > 0
                    ):
                        stock_leg = stock_exec_open / entry_adj
                        spy_leg = spy_terminal / spy_exec_open
                        strategy_return = stock_leg * spy_leg - 1.0
                    else:
                        continue
                else:
                    strategy_return = hold_return

                rows.append({
                    "ticker": ticker,
                    "entry_date": dates[e],
                    "entry_year": int(dates[e].year),
                    "raw_entry_close": float(entry_raw),
                    "threshold": float(threshold),
                    "horizon_label": label,
                    "horizon_days": int(h),
                    "hold_return": float(hold_return),
                    "strategy_return": float(strategy_return),
                    "delta": float(strategy_return - hold_return),
                    "triggered": int(triggered),
                    "trigger_date": dates[t] if triggered else pd.NaT,
                    "execution_date": dates[exec_pos] if triggered else pd.NaT,
                    "days_to_trigger": int(t - e) if triggered else np.nan,
                    "days_to_execution": int(exec_pos - e) if triggered else np.nan,
                })

    if not rows:
        return pd.DataFrame(columns=EVENT_COLUMNS)

    out = pd.DataFrame(rows)

    # Compact dtypes keep the event dataset manageable.
    out["ticker"] = out["ticker"].astype("category")
    out["horizon_label"] = out["horizon_label"].astype("category")
    out["entry_year"] = out["entry_year"].astype("int16")
    out["horizon_days"] = out["horizon_days"].astype("int16")
    out["triggered"] = out["triggered"].astype("int8")

    for c in ["raw_entry_close", "threshold", "hold_return", "strategy_return", "delta",
              "days_to_trigger", "days_to_execution"]:
        out[c] = pd.to_numeric(out[c], downcast="float")

    return out

In [ ]:
# Small deterministic unit test of the split-factor convention.
toy = pd.Series(
    [0.0, 0.0, 2.0, 0.0],
    index=pd.date_range("2020-01-01", periods=4)
)
toy_factor = _future_split_factor(toy)
assert np.allclose(toy_factor.to_numpy(), [2.0, 2.0, 1.0, 1.0])

# Yahoo-style split-adjusted closes around a 2:1 split on day 3: the series is continuous.
toy_yahoo_close = pd.Series([50.0, 51.0, 52.0, 53.0], index=toy.index)
toy_as_traded = toy_yahoo_close * toy_factor
assert np.allclose(toy_as_traded.to_numpy(), [100.0, 102.0, 52.0, 53.0])
print("Split-factor and as-traded reconstruction unit tests passed.")

In [ ]:
# Real-data check of the Yahoo split convention: NVDA 10-for-1, first split-adjusted session 2024-06-10.
# If Yahoo's Close were as-traded, split_close would fall ~10x across the split and this would fail.
nv = get_price_history("NVDA")
pre, post = pd.Timestamp("2024-06-07"), pd.Timestamp("2024-06-10")

if not nv.empty and pre in nv.index and post in nv.index:
    p = prepare_stock_on_spy_calendar(nv)
    display(p.loc["2024-06-05":"2024-06-12", ["raw_close", "split_close", "adj_close"]])

    split_move = p.loc[post, "split_close"] / p.loc[pre, "split_close"]
    as_traded_move = p.loc[pre, "raw_close"] / p.loc[post, "raw_close"]

    assert 0.8 < split_move < 1.25, f"split_close jumps {split_move:.2f}x across the split: convention is wrong"
    assert 7.0 < as_traded_move < 13.0, f"as-traded close should fall ~10x across a 10:1 split (got {as_traded_move:.2f}x)"
    print(f"NVDA check passed: split_close move {split_move:.3f}x, as-traded drop {as_traded_move:.2f}x.")
else:
    print("NVDA history unavailable around the split; real-data convention check skipped.")


## 6. Build transformed event dataset

The event table can be large, so it is written in Parquet parts and queried with DuckDB rather than kept entirely in memory.

These event files are still derived from Yahoo data and stay inside the ignored local work directory.

In [ ]:
if REBUILD_EVENTS:
    for p in EVENT_DIR.glob("part_*.parquet"):
        p.unlink()

existing_parts = sorted(EVENT_DIR.glob("part_*.parquet"))

if not existing_parts:
    part_frames = []
    part_tickers = 0
    part_id = 0
    total_rows = 0
    processed = 0

    for ticker in universe["ticker"]:
        raw = get_price_history(ticker)
        processed += 1

        if not raw.empty:
            ev = simulate_ticker(ticker, raw)
            if not ev.empty:
                part_frames.append(ev)
                total_rows += len(ev)

        part_tickers += 1

        if part_tickers >= 100 or processed == len(universe):
            if part_frames:
                block = pd.concat(part_frames, ignore_index=True)
                path = EVENT_DIR / f"part_{part_id:04d}.parquet"
                block.to_parquet(path, index=False)
                print(
                    f"Wrote {path.name}: {len(block):,} rows | "
                    f"processed {processed:,}/{len(universe):,} tickers"
                )
                part_id += 1

            part_frames = []
            part_tickers = 0

    print(f"Total generated rows: {total_rows:,}")
else:
    print(f"Using {len(existing_parts)} existing event parts. Set REBUILD_EVENTS=True to regenerate.")

In [ ]:
event_files = sorted(EVENT_DIR.glob("part_*.parquet"))
if not event_files:
    raise RuntimeError("No event files were generated.")

event_glob = (EVENT_DIR / "part_*.parquet").as_posix()
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW events AS
SELECT * FROM read_parquet('{event_glob}');
""")

dataset_overview = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT ticker) AS tickers,
    MIN(entry_date) AS first_entry,
    MAX(entry_date) AS last_entry,
    MIN(entry_year) AS first_year,
    MAX(entry_year) AS last_year
FROM events
""").df()

display(dataset_overview)

## 7. Primary summaries

Interpret `mean_delta` as:

**stop-and-switch-to-SPY return − buy-and-hold stock return**

- Positive: the stop rule did better.
- Negative: simply holding the stock did better.

In [ ]:
def event_weighted_summary(price_floor=0.0):
    return con.execute(f"""
    SELECT
        threshold,
        horizon_label,
        horizon_days,
        COUNT(*) AS n_events,
        COUNT(DISTINCT ticker) AS n_tickers,
        AVG(triggered)::DOUBLE AS trigger_rate,
        AVG(hold_return)::DOUBLE AS mean_hold_return,
        AVG(strategy_return)::DOUBLE AS mean_strategy_return,
        AVG(delta)::DOUBLE AS mean_delta,
        MEDIAN(delta)::DOUBLE AS median_delta,
        QUANTILE_CONT(delta, 0.05)::DOUBLE AS p05_delta,
        QUANTILE_CONT(delta, 0.25)::DOUBLE AS p25_delta,
        QUANTILE_CONT(delta, 0.75)::DOUBLE AS p75_delta,
        QUANTILE_CONT(delta, 0.95)::DOUBLE AS p95_delta,
        AVG(CASE WHEN delta > 0 THEN 1.0 ELSE 0.0 END)::DOUBLE AS stop_win_rate
    FROM events
    WHERE raw_entry_close >= {float(price_floor)}
    GROUP BY threshold, horizon_label, horizon_days
    ORDER BY horizon_days, threshold DESC
    """).df()

def stock_balanced_summary(price_floor=0.0):
    return con.execute(f"""
    WITH per_ticker AS (
        SELECT
            ticker,
            threshold,
            horizon_label,
            horizon_days,
            AVG(delta)::DOUBLE AS ticker_mean_delta,
            AVG(triggered)::DOUBLE AS ticker_trigger_rate
        FROM events
        WHERE raw_entry_close >= {float(price_floor)}
        GROUP BY ticker, threshold, horizon_label, horizon_days
    )
    SELECT
        threshold,
        horizon_label,
        horizon_days,
        COUNT(*) AS n_tickers,
        AVG(ticker_mean_delta)::DOUBLE AS mean_delta,
        MEDIAN(ticker_mean_delta)::DOUBLE AS median_delta,
        AVG(ticker_trigger_rate)::DOUBLE AS trigger_rate
    FROM per_ticker
    GROUP BY threshold, horizon_label, horizon_days
    ORDER BY horizon_days, threshold DESC
    """).df()

def year_balanced_summary(price_floor=0.0):
    return con.execute(f"""
    WITH per_year AS (
        SELECT
            entry_year,
            threshold,
            horizon_label,
            horizon_days,
            AVG(delta)::DOUBLE AS year_mean_delta,
            AVG(triggered)::DOUBLE AS year_trigger_rate
        FROM events
        WHERE raw_entry_close >= {float(price_floor)}
        GROUP BY entry_year, threshold, horizon_label, horizon_days
    )
    SELECT
        threshold,
        horizon_label,
        horizon_days,
        COUNT(*) AS n_years,
        AVG(year_mean_delta)::DOUBLE AS mean_delta,
        MEDIAN(year_mean_delta)::DOUBLE AS median_delta,
        AVG(year_trigger_rate)::DOUBLE AS trigger_rate
    FROM per_year
    GROUP BY threshold, horizon_label, horizon_days
    ORDER BY horizon_days, threshold DESC
    """).df()

def ticker_year_balanced_summary(price_floor=0.0):
    return con.execute(f"""
    WITH per_ticker_year AS (
        SELECT
            ticker,
            entry_year,
            threshold,
            horizon_label,
            horizon_days,
            AVG(delta)::DOUBLE AS ticker_year_mean_delta
        FROM events
        WHERE raw_entry_close >= {float(price_floor)}
        GROUP BY ticker, entry_year, threshold, horizon_label, horizon_days
    )
    SELECT
        threshold,
        horizon_label,
        horizon_days,
        COUNT(*) AS n_ticker_years,
        AVG(ticker_year_mean_delta)::DOUBLE AS mean_delta,
        MEDIAN(ticker_year_mean_delta)::DOUBLE AS median_delta
    FROM per_ticker_year
    GROUP BY threshold, horizon_label, horizon_days
    ORDER BY horizon_days, threshold DESC
    """).df()

event_summary = event_weighted_summary(0.0)
stock_summary = stock_balanced_summary(0.0)
year_summary = year_balanced_summary(0.0)
ticker_year_summary = ticker_year_balanced_summary(0.0)

display(event_summary)
display(stock_summary)
display(year_summary)
display(ticker_year_summary)

## 8. Cluster-aware bootstrap intervals

The event rows are highly correlated within ticker and calendar year. The main uncertainty checks therefore operate on already-aggregated ticker means and year means rather than pretending every monthly entry is independent.

In [ ]:
per_ticker = con.execute("""
SELECT
    ticker,
    threshold,
    horizon_label,
    horizon_days,
    AVG(delta)::DOUBLE AS mean_delta
FROM events
GROUP BY ticker, threshold, horizon_label, horizon_days
""").df()

per_year = con.execute("""
SELECT
    entry_year,
    threshold,
    horizon_label,
    horizon_days,
    AVG(delta)::DOUBLE AS mean_delta
FROM events
GROUP BY entry_year, threshold, horizon_label, horizon_days
""").df()

def bootstrap_mean_ci(values, n_boot=N_BOOT, seed=42):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    n = len(values)

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[b] = values[idx].mean()

    return float(values.mean()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))

def bootstrap_table(df, unit_name):
    rows = []
    for (threshold, label, h), g in df.groupby(["threshold", "horizon_label", "horizon_days"], observed=True):
        mean_, lo, hi = bootstrap_mean_ci(g["mean_delta"].to_numpy())
        rows.append({
            "threshold": float(threshold),
            "horizon_label": str(label),
            "horizon_days": int(h),
            "resampling_unit": unit_name,
            "n_units": int(len(g)),
            "mean_delta": mean_,
            "ci_low": lo,
            "ci_high": hi,
        })
    return pd.DataFrame(rows).sort_values(["horizon_days", "threshold"], ascending=[True, False])

ticker_boot = bootstrap_table(per_ticker, "ticker")
year_boot = bootstrap_table(per_year, "calendar_year")

bootstrap_results = pd.concat([ticker_boot, year_boot], ignore_index=True)
display(bootstrap_results)

## 9. Entry-price sensitivity: all, $1+, $5+

The filter is based on the as-traded close actually quoted on the entry date, reconstructed as Yahoo `Close` × later split ratios. It is not applied to today's split-adjusted equivalent.

In [ ]:
floor_rows = []

for floor in PRICE_FLOORS:
    e = event_weighted_summary(floor)
    s = stock_balanced_summary(floor)
    y = year_balanced_summary(floor)

    for name, tbl in [("event", e), ("stock", s), ("year", y)]:
        q = tbl[
            np.isclose(tbl["threshold"], PRIMARY_THRESHOLD) &
            (tbl["horizon_days"] == PRIMARY_HORIZON_DAYS)
        ].copy()
        if not q.empty:
            row = q.iloc[0].to_dict()
            row["price_floor"] = floor
            row["weighting"] = name
            floor_rows.append(row)

price_floor_sensitivity = pd.DataFrame(floor_rows)
display(price_floor_sensitivity)

## 10. Common-cohort comparison

Short horizons naturally contain more recent entries than the 3-year horizon. This sensitivity restricts every horizon to stock-month-threshold observations that also have a valid 3-year endpoint.

In [ ]:
common_cohort = con.execute("""
WITH cohort AS (
    SELECT DISTINCT ticker, entry_date, threshold
    FROM events
    WHERE horizon_days = 756
)
SELECT
    e.threshold,
    e.horizon_label,
    e.horizon_days,
    COUNT(*) AS n_events,
    COUNT(DISTINCT e.ticker) AS n_tickers,
    AVG(e.triggered)::DOUBLE AS trigger_rate,
    AVG(e.delta)::DOUBLE AS mean_delta,
    MEDIAN(e.delta)::DOUBLE AS median_delta
FROM events e
INNER JOIN cohort c
    ON e.ticker = c.ticker
   AND e.entry_date = c.entry_date
   AND e.threshold = c.threshold
GROUP BY e.threshold, e.horizon_label, e.horizon_days
ORDER BY e.horizon_days, e.threshold DESC
""").df()

display(common_cohort)

## 11. Era, annual, and rolling-start diagnostics

In [ ]:
era_summary = con.execute("""
SELECT
    CASE
        WHEN entry_year < 2000 THEN 'pre-2000'
        WHEN entry_year BETWEEN 2000 AND 2009 THEN '2000-2009'
        WHEN entry_year BETWEEN 2010 AND 2019 THEN '2010-2019'
        ELSE '2020+'
    END AS era,
    threshold,
    horizon_label,
    horizon_days,
    COUNT(*) AS n_events,
    COUNT(DISTINCT ticker) AS n_tickers,
    AVG(delta)::DOUBLE AS mean_delta,
    MEDIAN(delta)::DOUBLE AS median_delta,
    AVG(triggered)::DOUBLE AS trigger_rate
FROM events
GROUP BY era, threshold, horizon_label, horizon_days
ORDER BY horizon_days, threshold DESC, era
""").df()

annual_primary = con.execute(f"""
SELECT
    entry_year,
    COUNT(*) AS n_events,
    COUNT(DISTINCT ticker) AS n_tickers,
    AVG(delta)::DOUBLE AS mean_delta,
    MEDIAN(delta)::DOUBLE AS median_delta,
    AVG(triggered)::DOUBLE AS trigger_rate
FROM events
WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
  AND horizon_days = {PRIMARY_HORIZON_DAYS}
GROUP BY entry_year
ORDER BY entry_year
""").df()

annual_primary["rolling_10y_mean_delta"] = annual_primary["mean_delta"].rolling(10, min_periods=5).mean()

bounds = con.execute("SELECT MIN(entry_year), MAX(entry_year) FROM events").fetchone()
min_year, max_year = int(bounds[0]), int(bounds[1])
start_cutoffs = list(range(max(1990, min_year), max_year - 1, 5))

rolling_start_rows = []
for cutoff in start_cutoffs:
    q = con.execute(f"""
    SELECT
        COUNT(*) AS n_events,
        COUNT(DISTINCT ticker) AS n_tickers,
        AVG(delta)::DOUBLE AS mean_delta,
        MEDIAN(delta)::DOUBLE AS median_delta
    FROM events
    WHERE entry_year >= {cutoff}
      AND ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
      AND horizon_days = {PRIMARY_HORIZON_DAYS}
    """).df().iloc[0].to_dict()
    q["start_year"] = cutoff
    rolling_start_rows.append(q)

rolling_start = pd.DataFrame(rolling_start_rows)

display(
    era_summary[
        np.isclose(era_summary["threshold"], PRIMARY_THRESHOLD) &
        (era_summary["horizon_days"] == PRIMARY_HORIZON_DAYS)
    ]
)
display(annual_primary)
display(rolling_start)

## 12. Tail-risk and extreme-observation audit

Extreme observations are shown rather than silently winsorized away. If a result depends on a handful of implausible price histories, that should be visible before making a video claim.

In [ ]:
tail_summary = con.execute("""
SELECT
    threshold,
    horizon_label,
    horizon_days,
    COUNT(*) AS n_events,
    AVG(delta)::DOUBLE AS mean_delta,
    MEDIAN(delta)::DOUBLE AS median_delta,
    QUANTILE_CONT(delta, 0.01)::DOUBLE AS p01_delta,
    QUANTILE_CONT(delta, 0.05)::DOUBLE AS p05_delta,
    QUANTILE_CONT(delta, 0.95)::DOUBLE AS p95_delta,
    QUANTILE_CONT(delta, 0.99)::DOUBLE AS p99_delta,
    AVG(CASE WHEN delta <= -0.50 THEN 1.0 ELSE 0.0 END)::DOUBLE AS p_delta_le_minus_50pct,
    AVG(CASE WHEN delta >=  0.50 THEN 1.0 ELSE 0.0 END)::DOUBLE AS p_delta_ge_plus_50pct
FROM events
GROUP BY threshold, horizon_label, horizon_days
ORDER BY horizon_days, threshold DESC
""").df()

extreme_audit = con.execute("""
SELECT
    ticker,
    entry_date,
    raw_entry_close,
    threshold,
    horizon_label,
    hold_return,
    strategy_return,
    delta,
    triggered,
    trigger_date,
    execution_date
FROM events
WHERE raw_entry_close < 0.10
   OR raw_entry_close > 100000
   OR ABS(hold_return) > 50
   OR ABS(strategy_return) > 50
   OR ABS(delta) > 10
ORDER BY GREATEST(ABS(delta), ABS(hold_return), ABS(strategy_return)) DESC
LIMIT 100
""").df()

display(tail_summary)
display(extreme_audit)

## 13. Price-history quality checks

Some long-running ticker histories have discontinuities or ticker reuse that can distort the results. The checks below flag questionable histories at the ticker level and repeat the analysis on a hard-exclusion sample and a broader review-exclusion sample. Broad flags are not proof of bad data.


In [ ]:
# QC thresholds are deliberately explicit so the sensitivity analysis is reproducible.
# Factors are gross price ratios: 50x means a 4,900% one-session increase.
QC = {
    # Split-day consistency. Yahoo's Close is already split-adjusted, so close_t / close_{t-1}
    # across a correctly handled split should be near 1 (just the day's market move).
    # A move near 1/ratio instead means Yahoo missed the split; that gets its own hard flag.
    "review_split_dislocation": 5.0,
    "hard_split_dislocation": 20.0,

    # Split-adjusted one-session continuity.
    "review_one_day_up": 10.0,
    "review_one_day_down": 0.05,   # -95%; review only because true collapses can happen
    "hard_one_day_up": 50.0,

    # Potential ticker reuse / broken continuity after long data gaps.
    "gap_days": 180,
    "review_gap_up": 20.0,
    "review_gap_down": 0.05,
    "hard_gap_up": 100.0,
    "hard_gap_down": 0.01,

    # Event-level magnitudes are used as an additional history-level audit flag,
    # not as a row-by-row winsorization rule.
    "review_max_3y_hold_return": 100.0,   # >100x profit (+10,000%)
    "hard_max_3y_hold_return": 1000.0,   # >1000x profit (+100,000%)
    "review_max_abs_delta": 10.0,        # >1,000 percentage-point strategy gap
}

# Put independently verified bad/reused histories here, e.g. {"XXXX", "YYYY"}.
# Keep this empty unless you actually inspect the history.
MANUAL_EXCLUDE_TICKERS = set()

# v5: new cache paths so the QC table built with the old split check is never reused.
QC_HISTORY_CACHE = WORKDIR / "ticker_history_qc_base_v5.csv"
QC_AUDIT_PATH = WORKDIR / "ticker_history_qc_audit_v5.csv"
REBUILD_QC = False

QC

In [ ]:
def _split_event_checks(raw: pd.DataFrame) -> tuple:
    """
    Split-day checks on Yahoo's split-adjusted Close.

    Returns:
        max_dislocation: largest max(move, 1/move) on any split day, where move = close_t / close_{t-1}.
                         ~1 means the split was adjusted cleanly.
        n_missed:        number of material splits where the move is closer to 1/ratio than to 1,
                         i.e. Yahoo appears NOT to have applied the split adjustment.
    """
    if raw.empty:
        return np.nan, 0

    r = raw.copy().sort_index()
    close = pd.to_numeric(r["Close"], errors="coerce")
    split = pd.to_numeric(r["Stock Splits"], errors="coerce").fillna(0.0)

    vals = []
    n_missed = 0
    split_positions = np.flatnonzero(split.to_numpy() > 0)
    for pos in split_positions:
        if pos == 0:
            continue
        prev_c = close.iloc[pos - 1]
        curr_c = close.iloc[pos]
        ratio = split.iloc[pos]
        if not (np.isfinite(prev_c) and prev_c > 0 and np.isfinite(curr_c) and curr_c > 0 and ratio > 0):
            continue

        move = curr_c / prev_c
        if not (np.isfinite(move) and move > 0):
            continue
        vals.append(max(move, 1.0 / move))

        # Ignore tiny ratios (e.g. small stock dividends) where the two hypotheses are indistinguishable.
        material = abs(np.log(ratio)) >= np.log(1.25)
        if material and abs(np.log(move * ratio)) < abs(np.log(move)):
            n_missed += 1

    return (float(max(vals)) if vals else 1.0), int(n_missed)


def history_qc_metrics(ticker: str) -> dict:
    raw = get_price_history(ticker)
    base = {
        "ticker": ticker,
        "first_date": pd.NaT,
        "last_date": pd.NaT,
        "n_price_rows": 0,
        "n_nonpositive_raw_close": 0,
        "max_one_day_up_factor": np.nan,
        "min_one_day_factor": np.nan,
        "max_calendar_gap_days": np.nan,
        "max_long_gap_up_factor": np.nan,
        "min_long_gap_factor": np.nan,
        "max_split_dislocation_factor": np.nan,
        "n_missed_split_adjustments": 0,
    }
    if raw.empty:
        return base

    base["first_date"] = raw.index.min()
    base["last_date"] = raw.index.max()
    base["n_price_rows"] = int(len(raw))
    raw_close = pd.to_numeric(raw["Close"], errors="coerce")
    base["n_nonpositive_raw_close"] = int(((raw_close <= 0) & raw_close.notna()).sum())
    base["max_split_dislocation_factor"], base["n_missed_split_adjustments"] = _split_event_checks(raw)

    p = prepare_stock_on_spy_calendar(raw)
    s = pd.to_numeric(p["split_close"], errors="coerce")
    s = s[np.isfinite(s) & (s > 0)]
    if len(s) < 2:
        return base

    values = s.to_numpy(float)
    factors = values[1:] / values[:-1]
    gaps = np.diff(s.index.values).astype("timedelta64[D]").astype(int)

    session_like = gaps <= 7
    if session_like.any():
        sf = factors[session_like]
        base["max_one_day_up_factor"] = float(np.nanmax(sf))
        base["min_one_day_factor"] = float(np.nanmin(sf))

    base["max_calendar_gap_days"] = int(np.nanmax(gaps))

    long_gap = gaps >= QC["gap_days"]
    if long_gap.any():
        gf = factors[long_gap]
        base["max_long_gap_up_factor"] = float(np.nanmax(gf))
        base["min_long_gap_factor"] = float(np.nanmin(gf))

    return base


def build_ticker_qc():
    if QC_HISTORY_CACHE.exists() and not REBUILD_QC:
        out = pd.read_csv(QC_HISTORY_CACHE, parse_dates=["first_date", "last_date"])
        print(f"Loaded cached ticker QC: {len(out):,} tickers")
        return out

    rows = []
    tickers = universe["ticker"].astype(str).tolist()
    for i, ticker in enumerate(tickers, start=1):
        rows.append(history_qc_metrics(ticker))
        if i % 250 == 0 or i == len(tickers):
            print(f"QC history scan: {i:,}/{len(tickers):,}")

    out = pd.DataFrame(rows)
    out.to_csv(QC_HISTORY_CACHE, index=False)
    return out


ticker_qc = build_ticker_qc()

# Event-level extremes summarized to ticker level. These are audit signals, not row deletions.
event_qc = con.execute("""
SELECT
    ticker,
    MAX(hold_return)::DOUBLE AS max_hold_return,
    MIN(hold_return)::DOUBLE AS min_hold_return,
    MAX(ABS(delta))::DOUBLE AS max_abs_delta,
    MAX(CASE WHEN horizon_days = 252 THEN hold_return END)::DOUBLE AS max_1y_hold_return,
    MAX(CASE WHEN horizon_days = 756 THEN hold_return END)::DOUBLE AS max_3y_hold_return
FROM events
GROUP BY ticker
""").df()

ticker_qc = ticker_qc.merge(event_qc, on="ticker", how="left")
print(f"Ticker QC table: {len(ticker_qc):,} rows")

In [ ]:
# Build transparent flag columns.
ticker_qc["flag_nonpositive_hard"] = ticker_qc["n_nonpositive_raw_close"] > 0

ticker_qc["flag_split_review"] = ticker_qc["max_split_dislocation_factor"] >= QC["review_split_dislocation"]
ticker_qc["flag_split_hard"] = ticker_qc["max_split_dislocation_factor"] >= QC["hard_split_dislocation"]
ticker_qc["flag_split_missed_hard"] = ticker_qc["n_missed_split_adjustments"].fillna(0) > 0

ticker_qc["flag_day_up_review"] = ticker_qc["max_one_day_up_factor"] >= QC["review_one_day_up"]
ticker_qc["flag_day_down_review"] = ticker_qc["min_one_day_factor"] <= QC["review_one_day_down"]
ticker_qc["flag_day_up_hard"] = ticker_qc["max_one_day_up_factor"] >= QC["hard_one_day_up"]

ticker_qc["flag_gap_review"] = (
    (ticker_qc["max_calendar_gap_days"] >= QC["gap_days"]) &
    (
        (ticker_qc["max_long_gap_up_factor"] >= QC["review_gap_up"]) |
        (ticker_qc["min_long_gap_factor"] <= QC["review_gap_down"])
    )
)
ticker_qc["flag_gap_hard"] = (
    (ticker_qc["max_calendar_gap_days"] >= QC["gap_days"]) &
    (
        (ticker_qc["max_long_gap_up_factor"] >= QC["hard_gap_up"]) |
        (ticker_qc["min_long_gap_factor"] <= QC["hard_gap_down"])
    )
)

ticker_qc["flag_event_review"] = (
    (ticker_qc["max_3y_hold_return"] >= QC["review_max_3y_hold_return"]) |
    (ticker_qc["max_abs_delta"] >= QC["review_max_abs_delta"])
)
ticker_qc["flag_event_hard"] = ticker_qc["max_3y_hold_return"] >= QC["hard_max_3y_hold_return"]
ticker_qc["flag_manual"] = ticker_qc["ticker"].isin(MANUAL_EXCLUDE_TICKERS)

# Hard-clean: only severe continuity/extreme-history flags + manually verified exclusions.
ticker_qc["hard_exclude"] = ticker_qc[[
    "flag_nonpositive_hard",
    "flag_split_hard",
    "flag_split_missed_hard",
    "flag_day_up_hard",
    "flag_gap_hard",
    "flag_event_hard",
    "flag_manual",
]].any(axis=1)

# Broad-clean: deliberately aggressive sensitivity bound.
ticker_qc["review_exclude"] = ticker_qc["hard_exclude"] | ticker_qc[[
    "flag_split_review",
    "flag_day_up_review",
    "flag_day_down_review",
    "flag_gap_review",
    "flag_event_review",
    "flag_manual",
]].any(axis=1)


def _flag_reasons(row, broad=False):
    mapping = [
        ("flag_nonpositive_hard", "non-positive raw close"),
        ("flag_split_hard" if not broad else "flag_split_review", "split-day discontinuity"),
        ("flag_split_missed_hard", "split not adjusted by Yahoo"),
        ("flag_day_up_hard" if not broad else "flag_day_up_review", "one-session upward discontinuity"),
        ("flag_gap_hard" if not broad else "flag_gap_review", "long-gap/ticker-reuse discontinuity"),
        ("flag_event_hard" if not broad else "flag_event_review", "extreme multi-period event history"),
        ("flag_manual", "manual verified exclusion"),
    ]
    if broad and bool(row.get("flag_day_down_review", False)):
        mapping.append(("flag_day_down_review", "one-session collapse (review only)"))
    return "; ".join(label for col, label in mapping if bool(row.get(col, False)))


ticker_qc["hard_reasons"] = ticker_qc.apply(lambda r: _flag_reasons(r, broad=False), axis=1)
ticker_qc["review_reasons"] = ticker_qc.apply(lambda r: _flag_reasons(r, broad=True), axis=1)

# Keep this ticker-level derived QC file local/private with the rest of the Yahoo-derived work data.
ticker_qc.to_csv(QC_AUDIT_PATH, index=False)

analyzed = ticker_qc["max_hold_return"].notna()
raw_n = int(analyzed.sum())
hard_n = int((analyzed & ~ticker_qc["hard_exclude"]).sum())
review_n = int((analyzed & ~ticker_qc["review_exclude"]).sum())

qc_counts = pd.DataFrame({
    "sample": ["raw", "hard_clean", "broad_review_clean"],
    "n_tickers": [raw_n, hard_n, review_n],
    "n_excluded": [0, raw_n - hard_n, raw_n - review_n],
})
qc_counts["pct_tickers_excluded"] = qc_counts["n_excluded"] / raw_n if raw_n else np.nan

display(qc_counts)

### QC audit table

This is the table to inspect before deciding whether a particular ticker should be placed in `MANUAL_EXCLUDE_TICKERS`. The notebook **does not** silently remove every extreme winner. It shows the price-history behavior that caused the flag.

In [ ]:
qc_review_table = ticker_qc.loc[
    ticker_qc["hard_exclude"] | ticker_qc["review_exclude"],
    [
        "ticker", "first_date", "last_date", "n_price_rows",
        "max_split_dislocation_factor", "n_missed_split_adjustments",
        "max_one_day_up_factor", "min_one_day_factor",
        "max_calendar_gap_days", "max_long_gap_up_factor", "min_long_gap_factor",
        "max_1y_hold_return", "max_3y_hold_return", "max_abs_delta",
        "hard_exclude", "review_exclude", "hard_reasons", "review_reasons",
    ]
].copy()

# Put the most extreme histories first without converting this into an exclusion score.
qc_review_table["audit_rank"] = (
    np.log1p(qc_review_table["max_one_day_up_factor"].clip(lower=0).fillna(0)) +
    np.log1p(qc_review_table["max_split_dislocation_factor"].clip(lower=0).fillna(0)) +
    np.log1p(qc_review_table["max_3y_hold_return"].clip(lower=0).fillna(0)) +
    np.log1p(qc_review_table["max_abs_delta"].clip(lower=0).fillna(0))
)

qc_review_table = qc_review_table.sort_values(
    ["hard_exclude", "audit_rank"], ascending=[False, False]
)

display(qc_review_table.head(100))

### Create cleaned event views

`events_hard_clean` is the main cleaned sensitivity analysis. `events_review_clean` is intentionally more aggressive and should be presented as a bound, not as the preferred estimate.

In [ ]:
# Register the QC table with DuckDB and make cleaned views.
con.register("ticker_qc_df", ticker_qc[["ticker", "hard_exclude", "review_exclude"]])
con.execute("CREATE OR REPLACE TABLE ticker_qc_sql AS SELECT * FROM ticker_qc_df")

con.execute("""
CREATE OR REPLACE VIEW events_hard_clean AS
SELECT e.*
FROM events e
LEFT JOIN ticker_qc_sql q USING (ticker)
WHERE COALESCE(q.hard_exclude, FALSE) = FALSE
""")

con.execute("""
CREATE OR REPLACE VIEW events_review_clean AS
SELECT e.*
FROM events e
LEFT JOIN ticker_qc_sql q USING (ticker)
WHERE COALESCE(q.review_exclude, FALSE) = FALSE
""")

view_sizes = []
for sample, view in {
    "raw": "events",
    "hard_clean": "events_hard_clean",
    "broad_review_clean": "events_review_clean",
}.items():
    q = con.execute(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT ticker) AS n_tickers,
        COUNT(DISTINCT ticker || '|' || CAST(entry_date AS VARCHAR)) AS n_stock_month_entries
    FROM {view}
    """).df().iloc[0].to_dict()
    q["sample"] = sample
    view_sizes.append(q)

view_sizes = pd.DataFrame(view_sizes)[["sample", "n_rows", "n_tickers", "n_stock_month_entries"]]
display(view_sizes)

### Summaries across QC samples

Repeat the primary weighting and entry-price comparisons in each sample, including the $5+ ticker-year-balanced estimate.


In [ ]:
SAFE_EVENT_VIEWS = {"events", "events_hard_clean", "events_review_clean"}


def summary_from_view(view_name, price_floor=0.0, weighting="event"):
    if view_name not in SAFE_EVENT_VIEWS:
        raise ValueError(f"Unsupported view: {view_name}")

    floor = float(price_floor)

    if weighting == "event":
        sql = f"""
        SELECT
            threshold, horizon_label, horizon_days,
            COUNT(*) AS n_units,
            COUNT(DISTINCT ticker) AS n_tickers,
            AVG(triggered)::DOUBLE AS trigger_rate,
            AVG(delta)::DOUBLE AS mean_delta,
            MEDIAN(delta)::DOUBLE AS median_delta
        FROM {view_name}
        WHERE raw_entry_close >= {floor}
        GROUP BY threshold, horizon_label, horizon_days
        """

    elif weighting == "stock":
        sql = f"""
        WITH unit AS (
            SELECT ticker, threshold, horizon_label, horizon_days,
                   AVG(delta)::DOUBLE AS unit_delta,
                   AVG(triggered)::DOUBLE AS unit_trigger
            FROM {view_name}
            WHERE raw_entry_close >= {floor}
            GROUP BY ticker, threshold, horizon_label, horizon_days
        )
        SELECT threshold, horizon_label, horizon_days,
               COUNT(*) AS n_units,
               COUNT(*) AS n_tickers,
               AVG(unit_trigger)::DOUBLE AS trigger_rate,
               AVG(unit_delta)::DOUBLE AS mean_delta,
               MEDIAN(unit_delta)::DOUBLE AS median_delta
        FROM unit
        GROUP BY threshold, horizon_label, horizon_days
        """

    elif weighting == "year":
        sql = f"""
        WITH unit AS (
            SELECT entry_year, threshold, horizon_label, horizon_days,
                   AVG(delta)::DOUBLE AS unit_delta,
                   AVG(triggered)::DOUBLE AS unit_trigger
            FROM {view_name}
            WHERE raw_entry_close >= {floor}
            GROUP BY entry_year, threshold, horizon_label, horizon_days
        )
        SELECT threshold, horizon_label, horizon_days,
               COUNT(*) AS n_units,
               NULL::BIGINT AS n_tickers,
               AVG(unit_trigger)::DOUBLE AS trigger_rate,
               AVG(unit_delta)::DOUBLE AS mean_delta,
               MEDIAN(unit_delta)::DOUBLE AS median_delta
        FROM unit
        GROUP BY threshold, horizon_label, horizon_days
        """

    elif weighting == "ticker_year":
        sql = f"""
        WITH unit AS (
            SELECT ticker, entry_year, threshold, horizon_label, horizon_days,
                   AVG(delta)::DOUBLE AS unit_delta,
                   AVG(triggered)::DOUBLE AS unit_trigger
            FROM {view_name}
            WHERE raw_entry_close >= {floor}
            GROUP BY ticker, entry_year, threshold, horizon_label, horizon_days
        )
        SELECT threshold, horizon_label, horizon_days,
               COUNT(*) AS n_units,
               COUNT(DISTINCT ticker) AS n_tickers,
               AVG(unit_trigger)::DOUBLE AS trigger_rate,
               AVG(unit_delta)::DOUBLE AS mean_delta,
               MEDIAN(unit_delta)::DOUBLE AS median_delta
        FROM unit
        GROUP BY threshold, horizon_label, horizon_days
        """
    else:
        raise ValueError("weighting must be event, stock, year, or ticker_year")

    return con.execute(sql + " ORDER BY horizon_days, threshold DESC").df()


SAMPLE_VIEWS = {
    "raw": "events",
    "hard_clean": "events_hard_clean",
    "broad_review_clean": "events_review_clean",
}
WEIGHTINGS = ["event", "stock", "year", "ticker_year"]

full_qc_summary_parts = []
for sample, view in SAMPLE_VIEWS.items():
    for weighting in WEIGHTINGS:
        tmp = summary_from_view(view, 0.0, weighting)
        tmp["sample"] = sample
        tmp["weighting"] = weighting
        full_qc_summary_parts.append(tmp)

full_qc_summary = pd.concat(full_qc_summary_parts, ignore_index=True)

primary_qc_rows = []
for sample, view in SAMPLE_VIEWS.items():
    for floor in PRICE_FLOORS:
        for weighting in WEIGHTINGS:
            tmp = summary_from_view(view, floor, weighting)
            q = tmp[
                np.isclose(tmp["threshold"], PRIMARY_THRESHOLD) &
                (tmp["horizon_days"] == PRIMARY_HORIZON_DAYS)
            ]
            if q.empty:
                continue
            row = q.iloc[0].to_dict()
            row.update({
                "sample": sample,
                "price_floor": float(floor),
                "weighting": weighting,
            })
            primary_qc_rows.append(row)

primary_qc_comparison = pd.DataFrame(primary_qc_rows)
primary_qc_comparison["mean_delta_pp"] = 100 * primary_qc_comparison["mean_delta"]
primary_qc_comparison["median_delta_pp"] = 100 * primary_qc_comparison["median_delta"]

# Compact table most useful for the actual interpretation.
display(
    primary_qc_comparison[
        ["sample", "price_floor", "weighting", "n_units", "n_tickers", "trigger_rate", "mean_delta_pp", "median_delta_pp"]
    ].sort_values(["price_floor", "weighting", "sample"])
)

### Cluster-aware bootstrap after QC

The cleaned samples get the same ticker and calendar-year bootstrap treatment as the raw data.

In [ ]:
def cluster_means_from_view(view_name, cluster):
    if view_name not in SAFE_EVENT_VIEWS:
        raise ValueError(view_name)
    if cluster == "ticker":
        unit_cols = "ticker"
    elif cluster == "calendar_year":
        unit_cols = "entry_year"
    else:
        raise ValueError(cluster)

    return con.execute(f"""
    SELECT
        {unit_cols} AS cluster_id,
        threshold,
        horizon_label,
        horizon_days,
        AVG(delta)::DOUBLE AS mean_delta
    FROM {view_name}
    GROUP BY {unit_cols}, threshold, horizon_label, horizon_days
    """).df()


qc_boot_parts = []
for sample, view in SAMPLE_VIEWS.items():
    for cluster in ["ticker", "calendar_year"]:
        cm = cluster_means_from_view(view, cluster)
        bt = bootstrap_table(cm, cluster)
        bt["sample"] = sample
        qc_boot_parts.append(bt)

qc_bootstrap_results = pd.concat(qc_boot_parts, ignore_index=True)

primary_qc_bootstrap = qc_bootstrap_results[
    np.isclose(qc_bootstrap_results["threshold"], PRIMARY_THRESHOLD) &
    (qc_bootstrap_results["horizon_days"] == PRIMARY_HORIZON_DAYS)
].copy()
primary_qc_bootstrap[["mean_delta", "ci_low", "ci_high"]] *= 100
primary_qc_bootstrap = primary_qc_bootstrap.rename(columns={
    "mean_delta": "mean_delta_pp",
    "ci_low": "ci_low_pp",
    "ci_high": "ci_high_pp",
})

display(primary_qc_bootstrap.sort_values(["resampling_unit", "sample"]))

### Influence analysis: which tickers move the arithmetic mean?

This does not exclude anything. It answers whether the raw headline is broadly distributed or dominated by a few ticker histories.

In [ ]:
influence_primary = con.execute(f"""
SELECT
    ticker,
    COUNT(*) AS n_events,
    SUM(delta)::DOUBLE AS sum_delta,
    AVG(delta)::DOUBLE AS mean_delta,
    MAX(ABS(delta))::DOUBLE AS max_abs_delta
FROM events
WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
  AND horizon_days = {PRIMARY_HORIZON_DAYS}
GROUP BY ticker
""").df()

negative_total = -influence_primary.loc[influence_primary["sum_delta"] < 0, "sum_delta"].sum()
positive_total = influence_primary.loc[influence_primary["sum_delta"] > 0, "sum_delta"].sum()

most_negative = influence_primary.sort_values("sum_delta").head(25).copy()
most_positive = influence_primary.sort_values("sum_delta", ascending=False).head(25).copy()

most_negative["share_of_all_negative_contribution"] = (
    -most_negative["sum_delta"] / negative_total if negative_total > 0 else np.nan
)
most_positive["share_of_all_positive_contribution"] = (
    most_positive["sum_delta"] / positive_total if positive_total > 0 else np.nan
)

print("Largest negative contributors to raw 20% / 1y arithmetic mean:")
display(most_negative)
print("Largest positive contributors:")
display(most_positive)

### Distribution-robust diagnostics (secondary only)

Arithmetic mean return remains the primary economic comparison. Two secondary diagnostics show how sensitive the answer is to the extreme right tail:

- a **1% two-sided trimmed mean** of the event-level strategy advantage;
- the mean **log wealth ratio**, `log((1 + stop strategy return) / (1 + hold return))`, which measures proportional terminal-wealth differences without allowing a handful of giant winners to dominate as strongly.

These are diagnostics, not substitutes for the primary return analysis.

In [ ]:
def robust_diagnostics(view_name, price_floor=0.0, trim=0.01):
    if view_name not in SAFE_EVENT_VIEWS:
        raise ValueError(view_name)

    floor = float(price_floor)
    lo_q = float(trim)
    hi_q = float(1 - trim)

    # All heavy work remains inside DuckDB; no multi-million-row DataFrame is materialized.
    return con.execute(f"""
    WITH q AS (
        SELECT
            threshold, horizon_label, horizon_days,
            QUANTILE_CONT(delta, {lo_q})::DOUBLE AS trim_lo,
            QUANTILE_CONT(delta, {hi_q})::DOUBLE AS trim_hi
        FROM {view_name}
        WHERE raw_entry_close >= {floor}
        GROUP BY threshold, horizon_label, horizon_days
    )
    SELECT
        e.threshold, e.horizon_label, e.horizon_days,
        COUNT(*) AS n_events,
        AVG(CASE WHEN e.delta BETWEEN q.trim_lo AND q.trim_hi THEN e.delta END)::DOUBLE AS trimmed_mean_delta,
        AVG(
            CASE
                WHEN e.hold_return > -1 AND e.strategy_return > -1
                THEN LN(1 + e.strategy_return) - LN(1 + e.hold_return)
            END
        )::DOUBLE AS mean_log_wealth_ratio
    FROM {view_name} e
    INNER JOIN q
        ON e.threshold = q.threshold
       AND e.horizon_label = q.horizon_label
       AND e.horizon_days = q.horizon_days
    WHERE e.raw_entry_close >= {floor}
    GROUP BY e.threshold, e.horizon_label, e.horizon_days
    ORDER BY e.horizon_days, e.threshold DESC
    """).df()

robust_parts = []
for sample, view in SAMPLE_VIEWS.items():
    d = robust_diagnostics(view, 0.0, trim=0.01)
    d["sample"] = sample
    robust_parts.append(d)
robust_qc_diagnostics = pd.concat(robust_parts, ignore_index=True)

primary_robust = robust_qc_diagnostics[
    np.isclose(robust_qc_diagnostics["threshold"], PRIMARY_THRESHOLD) &
    (robust_qc_diagnostics["horizon_days"] == PRIMARY_HORIZON_DAYS)
].copy()
primary_robust["trimmed_mean_delta_pp"] = 100 * primary_robust["trimmed_mean_delta"]
primary_robust["mean_log_wealth_ratio_pct_approx"] = 100 * primary_robust["mean_log_wealth_ratio"]

display(primary_robust)

### Cleaned common-cohort, era, annual, and rolling-start checks

These repeat the most important time-structure analyses after QC so an apparent regime shift cannot be blamed solely on one broken historical series.

In [ ]:
def common_cohort_from_view(view_name):
    return con.execute(f"""
    WITH cohort AS (
        SELECT DISTINCT ticker, entry_date, threshold
        FROM {view_name}
        WHERE horizon_days = 756
    )
    SELECT
        e.threshold, e.horizon_label, e.horizon_days,
        COUNT(*) AS n_events,
        COUNT(DISTINCT e.ticker) AS n_tickers,
        AVG(e.triggered)::DOUBLE AS trigger_rate,
        AVG(e.delta)::DOUBLE AS mean_delta,
        MEDIAN(e.delta)::DOUBLE AS median_delta
    FROM {view_name} e
    INNER JOIN cohort c
        ON e.ticker = c.ticker
       AND e.entry_date = c.entry_date
       AND e.threshold = c.threshold
    GROUP BY e.threshold, e.horizon_label, e.horizon_days
    ORDER BY e.horizon_days, e.threshold DESC
    """).df()


def era_primary_from_view(view_name):
    return con.execute(f"""
    SELECT
        CASE
            WHEN entry_year < 2000 THEN 'pre-2000'
            WHEN entry_year BETWEEN 2000 AND 2009 THEN '2000-2009'
            WHEN entry_year BETWEEN 2010 AND 2019 THEN '2010-2019'
            ELSE '2020+'
        END AS era,
        COUNT(*) AS n_events,
        COUNT(DISTINCT ticker) AS n_tickers,
        AVG(delta)::DOUBLE AS mean_delta,
        MEDIAN(delta)::DOUBLE AS median_delta,
        AVG(triggered)::DOUBLE AS trigger_rate
    FROM {view_name}
    WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
      AND horizon_days = {PRIMARY_HORIZON_DAYS}
    GROUP BY era
    ORDER BY era
    """).df()


def annual_primary_from_view(view_name):
    return con.execute(f"""
    SELECT
        entry_year,
        COUNT(*) AS n_events,
        COUNT(DISTINCT ticker) AS n_tickers,
        AVG(delta)::DOUBLE AS mean_delta,
        MEDIAN(delta)::DOUBLE AS median_delta,
        AVG(triggered)::DOUBLE AS trigger_rate
    FROM {view_name}
    WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
      AND horizon_days = {PRIMARY_HORIZON_DAYS}
    GROUP BY entry_year
    ORDER BY entry_year
    """).df()


common_parts = []
era_parts = []
annual_parts = []
rolling_parts = []

for sample, view in SAMPLE_VIEWS.items():
    cc = common_cohort_from_view(view)
    cc["sample"] = sample
    common_parts.append(cc)

    er = era_primary_from_view(view)
    er["sample"] = sample
    era_parts.append(er)

    an = annual_primary_from_view(view)
    an["sample"] = sample
    an["rolling_10y_mean_delta"] = an["mean_delta"].rolling(10, min_periods=5).mean()
    annual_parts.append(an)

    if not an.empty:
        lo, hi = int(an["entry_year"].min()), int(an["entry_year"].max())
        for cutoff in range(max(1990, lo), hi - 1, 5):
            q = con.execute(f"""
            SELECT
                COUNT(*) AS n_events,
                COUNT(DISTINCT ticker) AS n_tickers,
                AVG(delta)::DOUBLE AS mean_delta,
                MEDIAN(delta)::DOUBLE AS median_delta
            FROM {view}
            WHERE entry_year >= {cutoff}
              AND ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
              AND horizon_days = {PRIMARY_HORIZON_DAYS}
            """).df().iloc[0].to_dict()
            q.update({"sample": sample, "start_year": cutoff})
            rolling_parts.append(q)

common_cohort_qc = pd.concat(common_parts, ignore_index=True)
era_qc = pd.concat(era_parts, ignore_index=True)
annual_qc = pd.concat(annual_parts, ignore_index=True)
rolling_start_qc = pd.DataFrame(rolling_parts)

display(era_qc)
display(rolling_start_qc)

### Cleaned tail-risk summary

In [ ]:
tail_qc_parts = []
for sample, view in SAMPLE_VIEWS.items():
    t = con.execute(f"""
    SELECT
        threshold, horizon_label, horizon_days,
        COUNT(*) AS n_events,
        AVG(delta)::DOUBLE AS mean_delta,
        MEDIAN(delta)::DOUBLE AS median_delta,
        QUANTILE_CONT(delta, 0.01)::DOUBLE AS p01_delta,
        QUANTILE_CONT(delta, 0.05)::DOUBLE AS p05_delta,
        QUANTILE_CONT(delta, 0.95)::DOUBLE AS p95_delta,
        QUANTILE_CONT(delta, 0.99)::DOUBLE AS p99_delta
    FROM {view}
    GROUP BY threshold, horizon_label, horizon_days
    ORDER BY horizon_days, threshold DESC
    """).df()
    t["sample"] = sample
    tail_qc_parts.append(t)

tail_qc_summary = pd.concat(tail_qc_parts, ignore_index=True)

display(
    tail_qc_summary[
        np.isclose(tail_qc_summary["threshold"], PRIMARY_THRESHOLD) &
        (tail_qc_summary["horizon_days"] == PRIMARY_HORIZON_DAYS)
    ]
)

### QC comparison plots

In [ ]:
# 20% stop / 1-year result across price floors, weighting schemes, and QC samples.
plot_primary = primary_qc_comparison.copy()
plot_primary["floor_label"] = plot_primary["price_floor"].map({0.0: "All", 1.0: "$1+", 5.0: "$5+"})

for weighting in WEIGHTINGS:
    g = plot_primary[plot_primary["weighting"] == weighting].copy()
    if g.empty:
        continue

    pivot = g.pivot(index="floor_label", columns="sample", values="mean_delta_pp").reindex(["All", "$1+", "$5+"])
    ax = pivot.plot(kind="bar", figsize=(8.5, 5.2))
    ax.axhline(0, linewidth=1)
    ax.set_xlabel("As-traded entry price filter")
    ax.set_ylabel("Stop-and-SPY advantage vs hold (percentage points)")
    ax.set_title(f"20% stop, 1-year horizon — {weighting.replace('_', '-')} weighting")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
# Annual raw vs cleaned comparison for the primary test.
fig, ax = plt.subplots(figsize=(10, 5.5))
for sample, g in annual_qc.groupby("sample", observed=True):
    g = g.sort_values("entry_year")
    ax.plot(g["entry_year"], 100 * g["mean_delta"], marker="o", label=sample)

ax.axhline(0, linewidth=1)
ax.set_xlabel("Entry year")
ax.set_ylabel("Stop-and-SPY advantage vs hold (percentage points)")
ax.set_title("20% stop, 1-year horizon: raw vs QC-cleaned annual results")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### QC-based survivorship tipping-point sensitivity

The original survivorship calculation is retained below. This version repeats it using the raw and cleaned **stock-balanced** results, because an outlier-sensitive survivor-only estimate should not be the sole basis of the tipping-point story.

In [ ]:
qc_survivorship_rows = []

for sample, view in SAMPLE_VIEWS.items():
    s = summary_from_view(view, 0.0, "stock")
    for _, r in s.iterrows():
        d = float(r["mean_delta"])
        for advantage in CATASTROPHIC_STOP_ADVANTAGES:
            if np.isfinite(d) and d < 0 and advantage > d:
                p = -d / (advantage - d)
            elif np.isfinite(d) and d >= 0:
                p = 0.0
            else:
                p = np.nan

            qc_survivorship_rows.append({
                "sample": sample,
                "threshold": float(r["threshold"]),
                "horizon_label": str(r["horizon_label"]),
                "horizon_days": int(r["horizon_days"]),
                "observed_stock_balanced_delta": d,
                "assumed_catastrophic_stop_advantage": advantage,
                "omitted_share_needed_to_reach_zero": p,
            })

qc_survivorship_tipping = pd.DataFrame(qc_survivorship_rows)

display(
    qc_survivorship_tipping[
        np.isclose(qc_survivorship_tipping["threshold"], PRIMARY_THRESHOLD) &
        (qc_survivorship_tipping["horizon_days"] == PRIMARY_HORIZON_DAYS)
    ]
)

## 14. Survivorship-bias tipping point

Suppose current-listed survivors show a mean stop advantage `D` (usually negative if stops underperform).

Assume the omitted catastrophic-failure cases would have favored the stop rule by `A` percentage points. The omitted share `p` required to make the combined mean exactly zero is:

\[
(1-p)D + pA = 0
\]

so

\[
p = \frac{-D}{A-D}.
\]

This does **not** estimate the true delisting rate. It answers a narrower question: *how much missing catastrophic-failure mass would be required to reverse the observed survivor-only conclusion?*

In [ ]:
survivorship_rows = []

for _, r in stock_summary.iterrows():
    d = float(r["mean_delta"])

    for advantage in CATASTROPHIC_STOP_ADVANTAGES:
        if np.isfinite(d) and d < 0 and advantage > d:
            p = -d / (advantage - d)
        elif np.isfinite(d) and d >= 0:
            p = 0.0
        else:
            p = np.nan

        survivorship_rows.append({
            "threshold": float(r["threshold"]),
            "horizon_label": str(r["horizon_label"]),
            "horizon_days": int(r["horizon_days"]),
            "observed_stock_balanced_delta": d,
            "assumed_catastrophic_stop_advantage": advantage,
            "omitted_share_needed_to_reach_zero": p,
        })

survivorship_tipping = pd.DataFrame(survivorship_rows)
display(survivorship_tipping)

## 15. Optional sector and industry analysis

This is off by default because Yahoo metadata calls are slow and prone to rate limits. If enabled, results are cached so subsequent runs do not repeat successful calls.

In [ ]:
METADATA_PATH = WORKDIR / "yf_metadata.csv"

def fetch_sector_industry(tickers):
    if METADATA_PATH.exists():
        meta = pd.read_csv(METADATA_PATH)
    else:
        meta = pd.DataFrame(columns=["ticker", "sector", "industry", "quoteType"])

    known = set(meta["ticker"].astype(str))
    new_rows = []

    for i, ticker in enumerate(tickers, start=1):
        if ticker in known:
            continue

        try:
            info = yf.Ticker(ticker).get_info()
            new_rows.append({
                "ticker": ticker,
                "sector": info.get("sector"),
                "industry": info.get("industry"),
                "quoteType": info.get("quoteType"),
            })
        except Exception:
            new_rows.append({
                "ticker": ticker,
                "sector": None,
                "industry": None,
                "quoteType": None,
            })

        if len(new_rows) >= 100:
            meta = pd.concat([meta, pd.DataFrame(new_rows)], ignore_index=True)
            meta.drop_duplicates("ticker", keep="last").to_csv(METADATA_PATH, index=False)
            new_rows = []

        if i % 100 == 0:
            print(f"metadata: {i:,}/{len(tickers):,}")

        time.sleep(METADATA_SLEEP_SECONDS)

    if new_rows:
        meta = pd.concat([meta, pd.DataFrame(new_rows)], ignore_index=True)

    meta = meta.drop_duplicates("ticker", keep="last")
    meta.to_csv(METADATA_PATH, index=False)
    return meta

if FETCH_SECTOR_INDUSTRY:
    metadata = fetch_sector_industry(universe["ticker"].tolist())

    ticker_primary = con.execute(f"""
    SELECT
        ticker,
        AVG(delta)::DOUBLE AS ticker_mean_delta,
        AVG(triggered)::DOUBLE AS ticker_trigger_rate,
        COUNT(*) AS n_entries
    FROM events
    WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
      AND horizon_days = {PRIMARY_HORIZON_DAYS}
    GROUP BY ticker
    """).df()

    joined = ticker_primary.merge(metadata, on="ticker", how="left")

    sector_summary = (
        joined.dropna(subset=["sector"])
        .groupby("sector", as_index=False)
        .agg(
            n_tickers=("ticker", "nunique"),
            mean_delta=("ticker_mean_delta", "mean"),
            median_delta=("ticker_mean_delta", "median"),
            mean_trigger_rate=("ticker_trigger_rate", "mean"),
        )
        .sort_values("mean_delta")
    )

    industry_summary = (
        joined.dropna(subset=["industry"])
        .groupby("industry", as_index=False)
        .agg(
            n_tickers=("ticker", "nunique"),
            mean_delta=("ticker_mean_delta", "mean"),
            median_delta=("ticker_mean_delta", "median"),
            mean_trigger_rate=("ticker_trigger_rate", "mean"),
        )
        .query("n_tickers >= 10")
        .sort_values("mean_delta")
    )

    display(sector_summary)
    display(industry_summary.head(30))
else:
    metadata = pd.DataFrame()
    sector_summary = pd.DataFrame()
    industry_summary = pd.DataFrame()
    print("Sector/industry metadata skipped. Set FETCH_SECTOR_INDUSTRY=True to run it.")

## 16. Figures


In [ ]:
# Stock-balanced stop advantage by stop threshold.
plot_df = stock_summary.copy()
plot_df["threshold_pct"] = -100 * plot_df["threshold"]
plot_df["mean_delta_pp"] = 100 * plot_df["mean_delta"]

fig, ax = plt.subplots(figsize=(9, 5.5))
for label, g in plot_df.groupby("horizon_label", observed=True):
    g = g.sort_values("threshold_pct")
    ax.plot(g["threshold_pct"], g["mean_delta_pp"], marker="o", label=str(label))

ax.axhline(0, linewidth=1)
ax.set_xlabel("Stop loss threshold (%)")
ax.set_ylabel("Stop-and-SPY advantage vs hold (percentage points)")
ax.set_title("Does cutting losses improve returns?")
ax.legend(title="Horizon")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Trigger frequency.
plot_df = event_summary.copy()
plot_df["threshold_pct"] = -100 * plot_df["threshold"]
plot_df["trigger_rate_pct"] = 100 * plot_df["trigger_rate"]

fig, ax = plt.subplots(figsize=(9, 5.5))
for label, g in plot_df.groupby("horizon_label", observed=True):
    g = g.sort_values("threshold_pct")
    ax.plot(g["threshold_pct"], g["trigger_rate_pct"], marker="o", label=str(label))

ax.set_xlabel("Stop loss threshold (%)")
ax.set_ylabel("Entries triggering stop (%)")
ax.set_title("How often does each stop get hit?")
ax.legend(title="Horizon")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Primary annual result with 10-year rolling mean.
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(annual_primary["entry_year"], 100 * annual_primary["mean_delta"], marker="o", label="Annual mean")
ax.plot(
    annual_primary["entry_year"],
    100 * annual_primary["rolling_10y_mean_delta"],
    linewidth=2,
    label="10-year rolling mean",
)
ax.axhline(0, linewidth=1)
ax.set_xlabel("Entry year")
ax.set_ylabel("Stop-and-SPY advantage vs hold (percentage points)")
ax.set_title("20% stop, 1-year horizon: stability through time")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Entry-price sensitivity for the primary 20% stop / 1-year horizon.
pf = price_floor_sensitivity.query("weighting == 'stock'").sort_values("price_floor").copy()
pf["label"] = pf["price_floor"].map({0.0: "All", 1.0: "$1+", 5.0: "$5+"})
pf["mean_delta_pp"] = 100 * pf["mean_delta"]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(pf["label"], pf["mean_delta_pp"])
ax.axhline(0, linewidth=1)
ax.set_xlabel("As-traded entry price filter")
ax.set_ylabel("Stock-balanced advantage vs hold (percentage points)")
ax.set_title("20% stop, 1-year horizon: penny-stock sensitivity")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

### Additional distributions and diagnostics

These figures show stop performance across thresholds and horizons, the range of stock-level outcomes, time to trigger, and the influence of extreme observations.


In [ ]:
def draw_annotated_heatmap(df, value_col, title, value_format="{:.1f}", multiplier=100.0, cbar_label="Percentage points"):
    horizon_order = ["1m", "3m", "6m", "1y", "3y"]
    tmp = df.copy()
    tmp["threshold_pct"] = (-100 * tmp["threshold"]).astype(int)
    pivot = (
        tmp.pivot(index="threshold_pct", columns="horizon_label", values=value_col)
        .reindex(index=sorted(tmp["threshold_pct"].unique()), columns=horizon_order)
    )

    values = multiplier * pivot.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(8.5, 5.8))
    im = ax.imshow(values, aspect="auto")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(list(pivot.columns))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"-{x}%" for x in pivot.index])
    ax.set_xlabel("Holding horizon")
    ax.set_ylabel("Stop threshold")
    ax.set_title(title)

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            val = values[i, j]
            if np.isfinite(val):
                ax.text(j, i, value_format.format(val), ha="center", va="center", fontsize=9)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap 1: stock-balanced stop advantage across thresholds and horizons.
draw_annotated_heatmap(
    stock_summary,
    value_col="mean_delta",
    title="Where stop losses help or hurt (stock-balanced)",
    value_format="{:.1f}",
    multiplier=100.0,
    cbar_label="Percentage points",
)

In [ ]:
# Heatmap 2: event-weighted trigger rate across thresholds and horizons.
draw_annotated_heatmap(
    event_summary,
    value_col="trigger_rate",
    title="How often the stop actually gets hit",
    value_format="{:.0f}%",
    multiplier=100.0,
    cbar_label="Percent of entries",
)

In [ ]:
# Figure: 20% stop across horizons, comparing raw and cleaned samples.
primary_horizon_story = full_qc_summary[
    np.isclose(full_qc_summary["threshold"], PRIMARY_THRESHOLD) &
    (full_qc_summary["weighting"] == "stock")
].copy()

fig, ax = plt.subplots(figsize=(8.8, 5.4))
for sample, g in primary_horizon_story.groupby("sample", observed=True):
    g = g.sort_values("horizon_days")
    ax.plot(g["horizon_label"], 100 * g["mean_delta"], marker="o", label=sample)

ax.axhline(0, linewidth=1)
ax.set_xlabel("Holding horizon")
ax.set_ylabel("Stop-and-SPY advantage vs hold (percentage points)")
ax.set_title("20% stop: does the conclusion change with horizon or QC cleaning?")
ax.legend(title="Sample")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Figure: distribution of event-level outcomes for the primary test.
def load_primary_delta(view_name):
    return con.execute(f"""
    SELECT delta
    FROM {view_name}
    WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
      AND horizon_days = {PRIMARY_HORIZON_DAYS}
    """).df()["delta"].astype(float).to_numpy()

raw_delta = load_primary_delta("events")
hard_delta = load_primary_delta("events_hard_clean")

combo = np.concatenate([raw_delta[np.isfinite(raw_delta)], hard_delta[np.isfinite(hard_delta)]])
lo, hi = np.quantile(combo, [0.01, 0.99])
raw_in = raw_delta[(raw_delta >= lo) & (raw_delta <= hi)]
hard_in = hard_delta[(hard_delta >= lo) & (hard_delta <= hi)]

fig, ax = plt.subplots(figsize=(8.8, 5.4))
ax.hist(raw_in, bins=80, alpha=0.45, density=True, label="raw")
ax.hist(hard_in, bins=80, alpha=0.45, density=True, label="hard_clean")
ax.axvline(0, linewidth=1)
ax.set_xlabel("Stop-and-SPY advantage vs hold (event-level return difference)")
ax.set_ylabel("Density")
ax.set_title("Distribution of outcomes: 20% stop, 1-year horizon")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Displayed middle 98% of events for readability: [{lo:.2f}, {hi:.2f}]")
print(f"Raw events outside displayed range: {(~((raw_delta >= lo) & (raw_delta <= hi))).sum():,}")
print(f"Hard-clean events outside displayed range: {(~((hard_delta >= lo) & (hard_delta <= hi))).sum():,}")

In [ ]:
# Figure: cumulative fraction of triggered stops over time.
def load_trigger_days(view_name):
    return con.execute(f"""
    SELECT days_to_execution
    FROM {view_name}
    WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
      AND horizon_days = {PRIMARY_HORIZON_DAYS}
      AND triggered = 1
      AND days_to_execution IS NOT NULL
    """).df()["days_to_execution"].astype(float).to_numpy()

fig, ax = plt.subplots(figsize=(8.8, 5.4))
for sample, view in [("raw", "events"), ("hard_clean", "events_hard_clean")]:
    vals = load_trigger_days(view)
    vals = vals[np.isfinite(vals)]
    vals.sort()
    y = np.arange(1, len(vals) + 1) / len(vals)
    ax.step(vals, y, where="post", label=sample)

ax.set_xlabel("Trading days from entry to stop execution")
ax.set_ylabel("Cumulative share of triggered events")
ax.set_title("How quickly do 20% stops get hit?")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Figure: top negative contributors to the raw arithmetic-mean headline.
neg = most_negative.head(15).sort_values("sum_delta", ascending=True).copy()
neg["sum_delta_pp"] = 100 * neg["sum_delta"]

fig, ax = plt.subplots(figsize=(8.8, 6.0))
ax.barh(neg["ticker"].astype(str), neg["sum_delta_pp"])
ax.set_xlabel("Total contribution to raw mean delta (percentage-point sum)")
ax.set_ylabel("Ticker")
ax.set_title("Which stocks most strongly push the headline against stop losses?")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Figure: top positive contributors to the raw arithmetic-mean headline.
pos = most_positive.head(15).sort_values("sum_delta", ascending=True).copy()
pos["sum_delta_pp"] = 100 * pos["sum_delta"]

fig, ax = plt.subplots(figsize=(8.8, 6.0))
ax.barh(pos["ticker"].astype(str), pos["sum_delta_pp"])
ax.set_xlabel("Total contribution to raw mean delta (percentage-point sum)")
ax.set_ylabel("Ticker")
ax.set_title("Which stocks most strongly push the headline in favor of stop losses?")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Figure: how many tickers are flagged by each QC rule?
qc_reason_counts = pd.DataFrame({
    "reason": [
        "non-positive raw close",
        "split-day discontinuity",
        "split not adjusted by Yahoo",
        "one-session upward discontinuity",
        "long-gap / ticker-reuse discontinuity",
        "extreme multi-period event history",
        "manual exclusions",
    ],
    "hard_clean": [
        int(ticker_qc["flag_nonpositive_hard"].sum()),
        int(ticker_qc["flag_split_hard"].sum()),
        int(ticker_qc["flag_split_missed_hard"].sum()),
        int(ticker_qc["flag_day_up_hard"].sum()),
        int(ticker_qc["flag_gap_hard"].sum()),
        int(ticker_qc["flag_event_hard"].sum()),
        int(ticker_qc["flag_manual"].sum()),
    ],
    "broad_review_clean": [
        int(ticker_qc["flag_nonpositive_hard"].sum()),
        int(ticker_qc["flag_split_review"].sum()),
        int(ticker_qc["flag_split_missed_hard"].sum()),
        int((ticker_qc["flag_day_up_review"] | ticker_qc["flag_day_down_review"]).sum()),
        int(ticker_qc["flag_gap_review"].sum()),
        int(ticker_qc["flag_event_review"].sum()),
        int(ticker_qc["flag_manual"].sum()),
    ],
})

qc_reason_plot = qc_reason_counts.set_index("reason")
ax = qc_reason_plot.plot(kind="bar", figsize=(10.0, 5.5))
ax.set_xlabel("QC reason")
ax.set_ylabel("Flagged tickers")
ax.set_title("Why are tickers excluded in the cleaned samples?")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

display(qc_reason_counts)

In [ ]:
# Figure: era-level comparison for the primary test.
era_plot = era_qc.copy()
era_plot["mean_delta_pp"] = 100 * era_plot["mean_delta"]
pivot = era_plot.pivot(index="era", columns="sample", values="mean_delta_pp")

ax = pivot.plot(kind="bar", figsize=(8.8, 5.4))
ax.axhline(0, linewidth=1)
ax.set_xlabel("Entry era")
ax.set_ylabel("Stop-and-SPY advantage vs hold (percentage points)")
ax.set_title("Does the stop-loss conclusion depend on market era?")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

### Results conditional on a 20% stop triggering

These summaries include only entries where a 20% stop actually executes before the one-year endpoint. They compare the frequency and magnitude of stop-strategy wins against the outcome of holding the original stock.

The `recovered_by_1y` measure means the stock's **adjusted buy-and-hold return** is non-negative at the endpoint. It is not a separate observation of an unadjusted share price crossing the original entry price.


In [ ]:
def triggered_primary_summary(view_name, price_floor=0.0):
    if view_name not in SAFE_EVENT_VIEWS:
        raise ValueError(view_name)

    floor = float(price_floor)
    return con.execute(f"""
    SELECT
        COUNT(*) AS n_triggered_events,
        COUNT(DISTINCT ticker) AS n_tickers,
        AVG(CASE WHEN delta > 0 THEN 1.0 ELSE 0.0 END)::DOUBLE AS sell_win_rate,
        AVG(delta)::DOUBLE AS mean_delta,
        MEDIAN(delta)::DOUBLE AS median_delta,
        AVG(hold_return)::DOUBLE AS mean_hold_return,
        MEDIAN(hold_return)::DOUBLE AS median_hold_return,
        AVG(strategy_return)::DOUBLE AS mean_strategy_return,
        MEDIAN(strategy_return)::DOUBLE AS median_strategy_return,
        AVG(CASE WHEN hold_return >= 0 THEN 1.0 ELSE 0.0 END)::DOUBLE AS recovered_by_1y,
        AVG(CASE WHEN hold_return >= 0.25 THEN 1.0 ELSE 0.0 END)::DOUBLE AS hold_return_ge_25pct,
        AVG(CASE WHEN hold_return >= 1.00 THEN 1.0 ELSE 0.0 END)::DOUBLE AS hold_return_ge_100pct
    FROM {view_name}
    WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
      AND horizon_days = {PRIMARY_HORIZON_DAYS}
      AND triggered = 1
      AND raw_entry_close >= {floor}
    """).df()

triggered_summary_parts = []
for sample, view in SAMPLE_VIEWS.items():
    for floor in [0.0, 5.0]:
        t = triggered_primary_summary(view, floor)
        t["sample"] = sample
        t["price_floor"] = floor
        triggered_summary_parts.append(t)

triggered_primary_summary_table = pd.concat(triggered_summary_parts, ignore_index=True)

for c in [
    "sell_win_rate", "mean_delta", "median_delta",
    "mean_hold_return", "median_hold_return",
    "mean_strategy_return", "median_strategy_return",
    "recovered_by_1y", "hold_return_ge_25pct", "hold_return_ge_100pct",
]:
    triggered_primary_summary_table[c + "_pct"] = 100 * triggered_primary_summary_table[c]

display(
    triggered_primary_summary_table[[
        "sample", "price_floor", "n_triggered_events", "n_tickers",
        "sell_win_rate_pct", "mean_delta_pct", "median_delta_pct",
        "mean_hold_return_pct", "mean_strategy_return_pct",
        "recovered_by_1y_pct", "hold_return_ge_25pct_pct", "hold_return_ge_100pct_pct",
    ]].sort_values(["price_floor", "sample"])
)

In [ ]:
# Video-friendly summary: among triggered 20% stops, how often does selling win,
# and how often does the stock eventually recover by the 1-year endpoint?
triggered_story = triggered_primary_summary_table[
    triggered_primary_summary_table["price_floor"].eq(0.0)
].copy()

plot = triggered_story.set_index("sample")[["sell_win_rate_pct", "recovered_by_1y_pct"]]
plot = plot.rename(columns={
    "sell_win_rate_pct": "Selling to SPY beats holding",
    "recovered_by_1y_pct": "Stock recovers to non-negative hold return",
})

ax = plot.plot(kind="bar", figsize=(9.0, 5.4))
ax.set_xlabel("QC sample")
ax.set_ylabel("Percent of triggered events")
ax.set_title("After a 20% stop fires: sell or wait for the rebound?")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
def triggered_rebound_buckets(view_name, price_floor=0.0):
    if view_name not in SAFE_EVENT_VIEWS:
        raise ValueError(view_name)

    floor = float(price_floor)
    return con.execute(f"""
    WITH x AS (
        SELECT
            *,
            CASE
                WHEN hold_return < 0 THEN 1
                WHEN hold_return < 0.25 THEN 2
                WHEN hold_return < 1.00 THEN 3
                ELSE 4
            END AS bucket_order,
            CASE
                WHEN hold_return < 0 THEN 'Still below entry at 1y'
                WHEN hold_return < 0.25 THEN 'Recovered: 0% to +25%'
                WHEN hold_return < 1.00 THEN 'Recovered: +25% to +100%'
                ELSE 'Recovered: >+100%'
            END AS rebound_bucket
        FROM {view_name}
        WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
          AND horizon_days = {PRIMARY_HORIZON_DAYS}
          AND triggered = 1
          AND raw_entry_close >= {floor}
    )
    SELECT
        bucket_order,
        rebound_bucket,
        COUNT(*) AS n_events,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS event_share,
        AVG(CASE WHEN delta > 0 THEN 1.0 ELSE 0.0 END)::DOUBLE AS sell_win_rate,
        AVG(delta)::DOUBLE AS mean_delta,
        MEDIAN(delta)::DOUBLE AS median_delta,
        AVG(hold_return)::DOUBLE AS mean_hold_return,
        MEDIAN(hold_return)::DOUBLE AS median_hold_return
    FROM x
    GROUP BY bucket_order, rebound_bucket
    ORDER BY bucket_order
    """).df()

rebound_bucket_parts = []
for sample, view in SAMPLE_VIEWS.items():
    r = triggered_rebound_buckets(view, 0.0)
    r["sample"] = sample
    rebound_bucket_parts.append(r)

triggered_rebound_buckets_table = pd.concat(rebound_bucket_parts, ignore_index=True)

for c in ["event_share", "sell_win_rate", "mean_delta", "median_delta", "mean_hold_return", "median_hold_return"]:
    triggered_rebound_buckets_table[c + "_pct"] = 100 * triggered_rebound_buckets_table[c]

display(triggered_rebound_buckets_table)

In [ ]:
# Figure: how common is each post-stop rebound outcome in the hard-clean sample?
rebound_hard = triggered_rebound_buckets_table[
    triggered_rebound_buckets_table["sample"].eq("hard_clean")
].sort_values("bucket_order")

fig, ax = plt.subplots(figsize=(9.0, 5.4))
ax.bar(rebound_hard["rebound_bucket"], rebound_hard["event_share_pct"])
ax.set_xlabel("1-year buy-and-hold outcome after the stop had triggered")
ax.set_ylabel("Share of triggered events (%)")
ax.set_title("What happens after a 20% stop is hit?")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Figure: does selling to SPY help or hurt within each rebound bucket?
rebound_effect = rebound_hard.set_index("rebound_bucket")[["mean_delta_pct", "median_delta_pct"]]
rebound_effect = rebound_effect.rename(columns={
    "mean_delta_pct": "Mean stop advantage",
    "median_delta_pct": "Median stop advantage",
})

ax = rebound_effect.plot(kind="bar", figsize=(9.0, 5.4))
ax.axhline(0, linewidth=1)
ax.set_xlabel("1-year buy-and-hold outcome after the stop had triggered")
ax.set_ylabel("Stop-and-SPY advantage vs hold (percentage points)")
ax.set_title("The trade-off: ordinary losers vs rebound winners")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Figure: within each rebound bucket, how often did selling actually beat holding?
fig, ax = plt.subplots(figsize=(9.0, 5.4))
ax.bar(rebound_hard["rebound_bucket"], rebound_hard["sell_win_rate_pct"])
ax.set_xlabel("1-year buy-and-hold outcome after the stop had triggered")
ax.set_ylabel("Events where selling to SPY won (%)")
ax.set_title("When does a 20% stop actually help?")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Figure: distribution of stop-vs-hold outcomes, conditional on the 20% stop firing.
def triggered_delta_values(view_name):
    if view_name not in SAFE_EVENT_VIEWS:
        raise ValueError(view_name)
    return con.execute(f"""
    SELECT delta
    FROM {view_name}
    WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
      AND horizon_days = {PRIMARY_HORIZON_DAYS}
      AND triggered = 1
    """).df()["delta"].astype(float).to_numpy()

triggered_delta_hard = triggered_delta_values("events_hard_clean")
triggered_delta_hard = triggered_delta_hard[np.isfinite(triggered_delta_hard)]

lo, hi = np.quantile(triggered_delta_hard, [0.01, 0.99])
shown = triggered_delta_hard[(triggered_delta_hard >= lo) & (triggered_delta_hard <= hi)]
mean_triggered_delta = triggered_delta_hard.mean()
median_triggered_delta = np.median(triggered_delta_hard)

fig, ax = plt.subplots(figsize=(9.0, 5.4))
ax.hist(shown, bins=80, density=True, alpha=0.7)
ax.axvline(0, linewidth=1, label="Break-even")
ax.axvline(mean_triggered_delta, linewidth=1.5, linestyle="--", label=f"Mean: {100*mean_triggered_delta:.1f} pp")
ax.axvline(median_triggered_delta, linewidth=1.5, linestyle=":", label=f"Median: {100*median_triggered_delta:.1f} pp")
ax.set_xlabel("Stop-and-SPY advantage vs hold")
ax.set_ylabel("Density")
ax.set_title("After the stop fires, outcomes are highly asymmetric")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Displayed middle 98% for readability: [{lo:.2f}, {hi:.2f}]")

In [ ]:
# Tail-mechanism sensitivity: how much does the average change as the biggest
# buy-and-hold rebounders are progressively removed from the triggered sample?
triggered_tail = con.execute(f"""
SELECT hold_return, delta
FROM events_hard_clean
WHERE ABS(threshold - ({PRIMARY_THRESHOLD})) < 1e-9
  AND horizon_days = {PRIMARY_HORIZON_DAYS}
  AND triggered = 1
  AND hold_return IS NOT NULL
  AND delta IS NOT NULL
""").df()

cut_fracs = [0.00, 0.001, 0.005, 0.01, 0.02, 0.05]
tail_sensitivity_rows = []

for frac in cut_fracs:
    if frac == 0:
        kept = triggered_tail
        cutoff = triggered_tail["hold_return"].max()
    else:
        cutoff = triggered_tail["hold_return"].quantile(1 - frac)
        kept = triggered_tail[triggered_tail["hold_return"] <= cutoff]

    tail_sensitivity_rows.append({
        "top_rebound_share_removed": frac,
        "hold_return_cutoff": cutoff,
        "n_events": len(kept),
        "mean_delta": kept["delta"].mean(),
        "median_delta": kept["delta"].median(),
        "sell_win_rate": (kept["delta"] > 0).mean(),
    })

tail_mechanism_sensitivity = pd.DataFrame(tail_sensitivity_rows)
tail_mechanism_sensitivity["share_removed_pct"] = 100 * tail_mechanism_sensitivity["top_rebound_share_removed"]
tail_mechanism_sensitivity["mean_delta_pp"] = 100 * tail_mechanism_sensitivity["mean_delta"]
tail_mechanism_sensitivity["median_delta_pp"] = 100 * tail_mechanism_sensitivity["median_delta"]

display(tail_mechanism_sensitivity)

fig, ax = plt.subplots(figsize=(8.8, 5.2))
ax.plot(tail_mechanism_sensitivity["share_removed_pct"], tail_mechanism_sensitivity["mean_delta_pp"], marker="o", label="Mean")
ax.plot(tail_mechanism_sensitivity["share_removed_pct"], tail_mechanism_sensitivity["median_delta_pp"], marker="o", label="Median")
ax.axhline(0, linewidth=1)
ax.set_xlabel("Largest buy-and-hold rebound events removed (%)")
ax.set_ylabel("Stop-and-SPY advantage vs hold (percentage points)")
ax.set_title("How much do the biggest rebounds drive the average?")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 17. Primary result table

The main comparison is displayed under each QC sample and weighting scheme, with the cluster bootstrap and tail diagnostics alongside it.


In [ ]:
# Final claim-ready comparison: raw and QC-cleaned estimates side by side.
primary_claim = primary_qc_comparison[
    (primary_qc_comparison["price_floor"].isin([0.0, 5.0])) &
    (primary_qc_comparison["weighting"].isin(["event", "stock", "year", "ticker_year"]))
].copy()

primary_claim = primary_claim[
    ["sample", "price_floor", "weighting", "n_units", "n_tickers", "trigger_rate", "mean_delta_pp", "median_delta_pp"]
].sort_values(["price_floor", "weighting", "sample"])

print("PRIMARY TEST: 20% close-based stop -> next open -> SPY, evaluated at 1 year")
print("Positive values favor cutting the loss; negative values favor holding.")
print()
display(primary_claim)

print("Cluster-aware 95% bootstrap intervals (percentage points):")
display(primary_qc_bootstrap.sort_values(["resampling_unit", "sample"]))

print("Secondary distribution-robust diagnostics:")
display(primary_robust[[
    "sample", "n_events", "trimmed_mean_delta_pp", "mean_log_wealth_ratio_pct_approx"
]])

print()
print("Interpretation rule:")
print("1) Lead with a conclusion only if raw, hard-clean, and broad-review-clean samples tell a coherent story.")
print("2) If the sign flips across QC samples or $5+ weighting schemes, make that heterogeneity the story.")
print("3) Do not describe broad-review exclusions as proven bad data; they are an aggressive sensitivity bound.")

## 18. Export summary tables

Aggregate tables are saved locally. The price histories, ticker-level QC files, and event-level Parquet files remain outside the repository.


In [ ]:
exports = {
    "event_weighted_summary.csv": event_summary,
    "stock_balanced_summary.csv": stock_summary,
    "year_balanced_summary.csv": year_summary,
    "ticker_year_balanced_summary.csv": ticker_year_summary,
    "bootstrap_results.csv": bootstrap_results,
    "price_floor_sensitivity.csv": price_floor_sensitivity,
    "common_cohort_summary.csv": common_cohort,
    "era_summary.csv": era_summary,
    "annual_primary.csv": annual_primary,
    "rolling_start_primary.csv": rolling_start,
    "tail_summary.csv": tail_summary,
    "survivorship_tipping_points.csv": survivorship_tipping,

    # QC v2 aggregate outputs.
    "qc_sample_counts.csv": qc_counts,
    "qc_view_sizes.csv": view_sizes,
    "qc_full_summary.csv": full_qc_summary,
    "qc_primary_comparison.csv": primary_qc_comparison,
    "qc_bootstrap_results.csv": qc_bootstrap_results,
    "qc_common_cohort.csv": common_cohort_qc,
    "qc_era_primary.csv": era_qc,
    "qc_annual_primary.csv": annual_qc,
    "qc_rolling_start_primary.csv": rolling_start_qc,
    "qc_tail_summary.csv": tail_qc_summary,
    "qc_robust_diagnostics.csv": robust_qc_diagnostics,
    "qc_survivorship_tipping_points.csv": qc_survivorship_tipping,
    "triggered_primary_summary.csv": triggered_primary_summary_table,
    "triggered_rebound_buckets.csv": triggered_rebound_buckets_table,
    "triggered_tail_mechanism_sensitivity.csv": tail_mechanism_sensitivity,
}

if not sector_summary.empty:
    exports["sector_summary.csv"] = sector_summary
if not industry_summary.empty:
    exports["industry_summary.csv"] = industry_summary

for filename, df in exports.items():
    path = AGG_DIR / filename
    df.to_csv(path, index=False)
    print("wrote", path)

print()
print("Public-safe aggregate directory:", AGG_DIR.resolve())
print("Keep private/local:", WORKDIR.resolve())
print("Ticker-level QC audit remains local at:", QC_AUDIT_PATH.resolve())